# Spectral Stacking JADES DR4 - DR5 Morphology Optical

In [ ]:
from astropy.table import table, hstack
from astropy.table import Column, join
import numpy as np
import re
from astropy.cosmology import LambdaCDM
import astropy.units as u
from astropy.cosmology import Planck18

# %matplotlib widget

import os
import glob
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec
import matplotlib.patches as patches
from joblib import Parallel, delayed


from astropy.coordinates import SkyCoord  # High-level coordinates
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
from astropy.coordinates import Angle, Latitude, Longitude  # Angles
import astropy.units as u

from astropy.io import fits
import glob
from astropy.visualization import ZScaleInterval
import astropy
from scipy import ndimage
import pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import linregress

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import Akima1DInterpolator
from scipy.interpolate import interp1d
from scipy import optimize as opt
from astropy import units as u
import sys
from astropy.table import Table
import emcee

# cosmology
from astropy.cosmology import Planck18 as cosmo  # You can change the cosmology model if needed
from astropy import units as u

# sampling spectra on common wavelength  grid:
from spectres import spectres

# nebular properties:
import pyneb as pn

# sigma clipping:
from astropy.stats import sigma_clip, mad_std


# mathplotlib aesthetic
# plt.rcParams['axes.linewidth'] = 1.5
# plt.rcParams.update({
#     "text.usetex": False,
#     "font.family": "STIXGeneral",  # Or 'DejaVu Serif', 'Latin Modern Roman'
# })

# plt.rcParams['xtick.labelsize'] = 12
# plt.rcParams['ytick.labelsize'] = 12
# plt.rcParams['axes.titlesize'] = 12  # Title font size
# plt.rcParams['axes.labelsize'] = 12  # Label font size

# plt.rcParams['text.usetex']= True
# plt.rcParams['mathtext.fontset']= 'custom'
# plt.rcParams['mathtext.default']= 'rm'
# plt.rcParams['axes.formatter.use_mathtext']=False
# colors
blue = "#5DD1EE"
violet = "#B375EA"
pink = "#FF65AB"
yellow = "#f7e618"
orange = "#ff8434"
green = "#40ba1b"

## JADES DR4 1D Spectra DATA

### Finding catalog and sampling

In [ ]:
filename = "DR4/Combined_DR4_external_v1.2.1.fits" # DR4 JADES sources
with fits.open(filename) as hdu:
    # hdu.info()
    obs = hdu[1].data # this is observational data (obs_info)
    data = hdu[3].data # this is prism R100 3 pixels extension (R1003pix)
    tab_obs = Table(obs)
    tab_data = Table(data)
    # tab.sort("Unique_ID")
print(f"Number of sources in JADES DR4: {len(tab_obs)}")
joined = join(tab_obs, tab_data, keys='Unique_ID', join_type='inner') # joining the two tables

# ----- crossmatched morphology catalog ------- (trying F115W, F444W filters):
tab_f115w = table.Table.read("Crossmatched_Tables/DR4_F115W_bestfit_stellar_prop.csv", delimiter = ",")
tab_f444w = table.Table.read("Crossmatched_Tables/DR4_F444W_bestfit.csv", delimiter = ",")

print(len(tab_f115w), len(tab_f444w))

# ---------- only looking prism medium hst, jwst ------------:

# wanted_tiers = ["goods-n-mediumjwst", "goods-n-mediumhst"]
# mask = np.isin(joined["TIER_1"], wanted_tiers)
# tab = joined[mask]

# print(f"Number of sources in JADES DR4: {len(tab)}")

### Cuts on the Dataset: 

In [ ]:
# ------------- removing bad redshift sources ---------:

tab1 = tab_f115w[tab_f115w["z_Spec_flag"]!="D"]
tab2 = tab1[tab1["z_Spec_flag"]!="E"]
# tab3 = tab2[tab2["z_Spec"]>=2]
tab3 = tab2
print(f"No. of spectroscopically confirmed sources matched: {len(tab3)}")

# --------- having enough emission lines check ----------:

O3_combined = np.where(np.isfinite(tab3["O3_5007d_flux"]), tab3["O3_5007d_flux"], tab3["O3_5007_flux"])
O3_err_combined = np.where(np.isfinite(tab3["O3_5007d_flux_err"]), tab3["O3_5007d_flux_err"], tab3["O3_5007_flux_err"])
tab3["O3_5007_combined_flux"] = O3_combined
tab3["O3_5007_combined_flux_err"] = O3_err_combined
tab4 = tab3[(~np.isnan(tab3["O3_5007_combined_flux"])) & (~np.isnan(tab3["O3_5007_combined_flux_err"]))]
print(f"No. of spectroscopically confirmed sources with emission lines: {len(tab4)}")

max_redshift = np.max(tab4["z_Spec_1"])
min_redshift = np.min(tab4["z_Spec_1"])
print(f"Max Redshift = {format(max_redshift, ".5f")}")
print(f"Min Redshift = {format(min_redshift, ".5f")}")

# ---- removing sources with incomplete spectra ----- :
# remove_ids = [1010202, 113461, 1011601, 109645, 206146, 205654, 202385, 146333,
#               1008580, 1005088, 429578, 140125,
#               1089840, 206996,
#               1001254, 1010544, 1007424, 1009104, 1018536, 25173, 1030362,
#               1088876, 116898, 121233, 137573, 205144]

remove_ids = [1008580, 1005088, 429578, 146333, 140125, 
              1089840, 205654, 202385, 206996, 1001254, 
              1010544, 1007424, 1009104, 113461, 1018536, 25173, 1030362, 
              1088876, 1011601, 121233, 137573, 206146, 205144]

tab4 = tab4[~np.isin(tab4["ID"], remove_ids)]

# tab4 = tab3[(tab3["z_Spec"]>=4) & (tab3["z_Spec"]<6)]
# print(f"No. of spectroscopically confirmed sources matched with z>=4: {len(tab4)}")

# ----------- S/N cut --------:
# snr_O3 = tab4["O3_5007_combined_flux"]/tab4["O3_5007_combined_flux_err"]
# tab4["SNR_O3_5007"] = snr_O3
# tab_prism = tab4[tab4["SNR_O3_5007"]>=0]
# print(f"No. of spectroscopically confirmed sources with high S/N ([OIII 5007]) emission lines: {len(tab_prism)}")

# tab_prism = tab5

### 1D Fits Files from DR4

In [ ]:
prism_goods_n_jwst = glob.glob("DR4/goods-n/clear-prism/goods-n-mediumjwst/*") # goods-n field, JWST 
prism_goods_n_hst = glob.glob("DR4/goods-n/clear-prism/goods-n-mediumhst/*") # goods-n field, HST 

prism_goods_s_deephst = glob.glob("DR4/goods-s/clear-prism/goods-s-deephst/*") # goods-n field, medium HST 
prism_goods_s_deepjwst = glob.glob("DR4/goods-s/clear-prism/goods-s-deepjwst/*") # goods-n field, medium HST 
prism_goods_s_mediumhst = glob.glob("DR4/goods-s/clear-prism/goods-s-mediumhst/*") # goods-n field, medium HST 
prism_goods_s_mediumjwst = glob.glob("DR4/goods-s/clear-prism/goods-s-mediumjwst/*") # goods-n field, medium JWST 
prism_goods_s_ultradeep = glob.glob("DR4/goods-s/clear-prism/goods-s-ultradeep/*") # goods-n field, medium HST 

prism_goods_field = prism_goods_n_jwst + prism_goods_n_hst + prism_goods_s_deephst + prism_goods_s_deepjwst + prism_goods_s_mediumhst + prism_goods_s_mediumjwst + prism_goods_s_ultradeep
prism_goods_field = np.sort(prism_goods_field)


print(f"No. of medium HST 1D files in GOODS-N field: {len(prism_goods_n_hst)}")
print(f"No. of medium JWST 1D files in GOODS-N field: {len(prism_goods_n_jwst)}")

print(f"No. of deep HST 1D files in GOODS-S field: {len(prism_goods_s_deephst)}")
print(f"No. of deep JWST 1D files in GOODS-S field: {len(prism_goods_s_deepjwst)}")  
print(f"No. of medium HST 1D files in GOODS-S field: {len(prism_goods_s_mediumhst)}")
print(f"No. of medium JWST 1D files in GOODS-S field: {len(prism_goods_s_mediumjwst)}")
print(f"No. of total ultradeep 1D files in GOODS-S field: {len(prism_goods_s_ultradeep)}")      

prism_ids = []
prism_goods_field = np.sort(prism_goods_field)
for i, file in enumerate(prism_goods_field):
    ID = int(file[-34:-26])
    prism_ids.append(ID)
    # print(ID)
print(f"No. of total 1D files in GOODS field: {len(prism_goods_field)}")      


### Finding Duplicates and Flagging

In [ ]:
# finding duplicate sources in Prism observations of GOODS-N:

ids, counts = np.unique(prism_ids, return_counts=True)
duplicates = ids[counts > 1]

print(f"Number of duplicate IDs: {len(duplicates)}")
print("Duplicate NIRSpec IDs:", duplicates)
duplicates = np.array(duplicates)     


In [ ]:
# flagging and removing the duplicates:

prism_goods_unique = []
prism_ids_unique = []
for i, file in enumerate(prism_goods_field):
    ID = int(file[-34:-26])
    if ID not in duplicates:
        prism_goods_unique.append(file)
        prism_ids_unique.append(ID)
        
print(len(prism_goods_field), len(prism_goods_unique))

### Matching Files to the Catalog:

In [ ]:
# ----------- removing sources which have duplicates in the field ---------:

tab_prism = tab4[~np.isin(tab4["NIRSpec_ID_1"], duplicates)]
print(f"No. of Prism sources after removing flags: {len(tab_prism)}")

# -------- matching the tab_prism observations to the files ---------:

prism_id_to_file = {int(f[-34:-26]): f for f in prism_goods_unique}
tab_ids = tab_prism["NIRSpec_ID_1"].astype(int)
mask_has_prism = np.isin(tab_ids, list(prism_id_to_file.keys()))
tab_prism_ = tab_prism[mask_has_prism]
print(f"No. of Prism files matched with table with confirmed redshifts: {len(tab_prism_)}")

# ------- adding the filenames to the table ------:

filenames = []
for ID in tab_prism_["NIRSpec_ID_1"]:
    match = [f for f in prism_goods_unique if int(f[-34:-26]) == ID] # find the matching file
    filenames.append(match[0] if len(match) > 0 else "")
    # filenames.append(match)
tab_prism_['file_prism'] = filenames
tab_prism_ = tab_prism_[tab_prism_["file_prism"]!=""]
prism_goods_files = tab_prism_['file_prism']
print(f"No. of Prism sources in table: {len(filenames)}")
tab_prism_

### Compactness Calculations

In [ ]:
arcsec_per_kpc = Planck18.arcsec_per_kpc_proper(tab_prism_["z_Spec_1"])
val = tab_prism_["R_EFF"]/(arcsec_per_kpc)

D = cosmo.angular_diameter_distance(tab_prism_["z_Spec_1"])
d = (tab_prism_["R_EFF"]*D)/206265

tab_prism_["Physical_Radius"] = d # in Mpc
tab_prism_["Physical_Radius_kpc"] = d*1e3

# tab_final[tab_final["Physical_Radius"]<=200*1e-6]
# print(arcsec_per_kpc)

print(min(tab_prism_["Physical_Radius"]*1e3), max(tab_prism_["Physical_Radius"]*1e3) ,"kpc")
print(np.mean(tab_prism_["Physical_Radius"]*1e3), "kpc")
# tab_prism_

### Redshift Data in Catalog

In [ ]:
z_low = 4
z_high = 7

max_redshift = np.max(tab_prism_["z_Spec_1"])
min_redshift = np.min(tab_prism_["z_Spec_1"])

print(f"Max Redshift = {format(max_redshift, ".5f")}")
print(f"Min Redshift = {format(min_redshift, ".5f")}")
tab_final = tab_prism_[(tab_prism_["z_Spec_1"]>=z_low) & (tab_prism_["z_Spec_1"]<=z_high)]
# tab_final = tab_prism_

# tab_final.write("Crossmatched_Tables/tab_final.csv", format="csv", overwrite=True)
prism_goods_n_files = tab_final["file_prism"]
print(f"No. of Prism sources in table (4<z<7): {len(prism_goods_n_files)}")

max_redshift = np.max(tab_final["z_Spec_1"])
min_redshift = np.min(tab_final["z_Spec_1"])

print(f"Max Redshift = {format(max_redshift, ".5f")}")
print(f"Min Redshift = {format(min_redshift, ".5f")}")

# compactness criterion (val = d)

arcsec_per_kpc = Planck18.arcsec_per_kpc_proper(tab_final["z_Spec_1"])
val = tab_final["R_EFF"]/(arcsec_per_kpc)

D = cosmo.angular_diameter_distance(tab_final["z_Spec_1"])
d = (tab_final["R_EFF"]*D)/206265

tab_final["Physical_Radius"] = d # in Mpc

# tab_final[tab_final["Physical_Radius"]<=200*1e-6]
# print(arcsec_per_kpc)

print(min(tab_final["Physical_Radius"]*1e6), max(tab_final["Physical_Radius"]*1e6) ,"pc")
print(np.mean(tab_final["Physical_Radius"]*1e6), "pc")
# tab_final


### Compactness Criterion:

In [ ]:
plt.figure(figsize = (5,3), dpi=200)
plt.scatter(tab_prism_["z_Spec_1"],tab_prism_["Physical_Radius_kpc"], marker = ".", s=20, alpha=0.3)
plt.xlabel("Redshift ($z$)")
plt.ylabel("R$_{eff}$ (kpc)")
plt.show()

plt.figure(figsize = (5,3), dpi=200)
plt.scatter(tab_final["z_Spec_1"],tab_final["Physical_Radius_kpc"], marker = ".", s=20, alpha=0.6, color = blue)
plt.xlabel("Redshift ($z$)")
plt.ylabel("R$_{eff}$ (kpc)")
plt.show()

# --------- Morishita 2024 Fig 6 --------:
M_o = 1e8
alpha = 0.19 # based on relation of R_eff proportional to M_*^0.19 +/- 0.03
stellar_mass = np.power(10,tab_final["log(Mstar)"])
stellar_mass_norm = (stellar_mass/M_o)**alpha

size = (tab_final["Physical_Radius_kpc"]/stellar_mass_norm)
log_size = np.log10(size)
tab_final["log_size"] = log_size
plt.figure(figsize = (5,3), dpi=200)
plt.scatter((tab_final["z_Spec_1"]),np.log10(size), marker = ".", s=20, alpha=0.6, color = blue)
plt.xlabel("Redshift ($z$)")
plt.ylabel(r"$\log(R_{eff}/(M_{*}/M_\odot)^{\alpha})$ (kpc)")
# plt.ylim(-2,)
plt.show()


# ------- median sizes in redshift bins + polyfit ------- :
z = tab_final["z_Spec_1"]
y = np.log10(size)   # log mass-corrected size

z_bins = np.arange(z_low, z_high + 1)

z_med = []
y_med = [] # median/mean
y_low = [] # 16th percentile
y_high = [] # 84th percentile 
y_sigma = [] # sigma = (84th - 16th)/2

for i in range(len(z_bins)-1):

    mask = (z >= z_bins[i]) & (z < z_bins[i+1])

    if np.sum(mask) > 0:

        z_med.append(np.median(z[mask]))
        y_med.append(np.median(y[mask]))

        # scatter (16–84 percentile like paper)
        y_low.append(np.percentile(y[mask],16))
        y_high.append(np.percentile(y[mask],84))
        y_sigma.append(0.5*((np.percentile(y[mask],84))-(np.percentile(y[mask],16))))

x_fit = np.log10(1 + np.array(z_med)) # uses the median z values in each bin
y_fit = np.array(y_med)

# alpha_z, intercept = np.polyfit(x_fit, y_fit, 1) 

fit = linregress(x_fit, y_fit) # for errors

alpha_z = fit.slope
alpha_z_err = fit.stderr
intercept = fit.intercept

print(rf"$\alpha_z = {alpha_z:.2f} \pm {alpha_z_err:.2f}$")
# ----- plot ----- :

plt.figure(figsize=(5,3), dpi=200)
plt.scatter(z, y, s=10, alpha=0.6, color="grey")

# median points
colors = [blue, violet, pink, orange, yellow, green]

for i in range(len(z_med)):
    plt.scatter(z_med[i], y_med[i],s=50,color=colors[i],edgecolor="black",zorder=5)
    plt.axvspan(z_bins[i],z_bins[i+1], zorder = 5, color=colors[i], alpha=0.1)

# error bars
plt.errorbar(z_med, y_med, yerr=[np.array(y_med)-np.array(y_low), np.array(y_high)-np.array(y_med)],
             fmt="none", color="black", capsize=3)

# power law fit
z_line = np.linspace(z_bins[0],z_bins[-1],200)
y_model = intercept + alpha_z*np.log10(1+z_line)

plt.plot(z_line, y_model, color="black",lw=1, label = rf"$R_{{\rm e}} \propto (1+z)^{{{alpha_z:.2f} \pm {alpha_z_err:.2f}}}$")

plt.xlabel("Redshift ($z$)")
plt.ylabel(r"$\log (R_{e}/(M_{*}/M_{0})^{\alpha})$ [kpc]")
plt.legend()
plt.show()

In [ ]:
compact = []
extended = []
intermediate = []
for row in tab_final:
    redshift = row["z_Spec_1"]
    size = row["log_size"]

    for i in range(len(z_bins)-1):
        if z_bins[i] <= redshift < z_bins[i+1]:
        
            # if size <= y_med[i] - 1*y_sigma[i]:
            if size < y_low[i]:
                compact.append(list(row))
            # elif size >= y_med[i] + 1*y_sigma[i]:
            elif size > y_high[i]:
                extended.append(list(row))
            else:
                intermediate.append(list(row))
            break

compact = Table(rows=compact, names=tab_final.colnames)
extended = Table(rows=extended, names=tab_final.colnames)
intermediate = Table(rows=intermediate, names=tab_final.colnames)

print(len(compact), len(extended), len(intermediate))

plt.figure(figsize=(5,3), dpi=200)
plt.scatter(z, y, s=10, alpha=0.6, color="grey")

# median points
colors = [blue, violet, pink, orange, yellow, green]

for i in range(len(z_med)):
    plt.scatter(z_med[i], y_med[i],s=30,color="black",edgecolor="black",zorder=5)
    plt.axvspan(z_bins[i],z_bins[i+1], zorder = 5, color=colors[i], alpha=0.1)

# error bars
plt.errorbar(z_med, y_med, yerr=[np.array(y_med)-np.array(y_low), np.array(y_high)-np.array(y_med)],
             fmt="none", color="black", capsize=3)

# power law fit
z_line = np.linspace(z_bins[0],z_bins[-1],200)
y_model = intercept + alpha_z*np.log10(1+z_line)

plt.plot(z_line, y_model, color="black",lw=1, label = rf"$R_{{\rm e}} \propto (1+z)^{{{alpha_z:.2f} \pm {alpha_z_err:.2f}}}$")
plt.scatter(compact["z_Spec_1"], compact["log_size"],
            s=15, color="#31ACC4", label="Compact")

plt.scatter(intermediate["z_Spec_1"], intermediate["log_size"],
            s=15, color="#F5C242", label="Intermediate")

plt.scatter(extended["z_Spec_1"], extended["log_size"],
            s=15, color="firebrick", label="Extended")

plt.xlabel("Redshift ($z$)")
plt.ylabel(r"$\log$ $(R_{e}/(M_{*}/M_{0})^{\alpha})$ [kpc]")
plt.legend()
plt.show()

mask = (compact["z_Spec_1"] > 7) & (compact["z_Spec_1"] <= 9)
weird_sources = compact[mask]
for ID in weird_sources["ID"]:
    print(ID)

## Weird Sources Spectra

In [ ]:
z_med = np.nanmedian(weird_sources["z_Spec_1"])

# --- rest-frame wave grid ---
wave_common_weird_sources = make_prism_rest_wave_grid(lam_min_rest=0.1, lam_max_rest=0.8, z_med=z_med, R_interp=R_interp)

for i,row in enumerate(weird_sources[:]):

    print(row["file_prism"])
    z = row["z_Spec_1"]
    with fits.open(row["file_prism"]) as hdu:
        # hdu.info()
        tab = Table(hdu[2].data) # using the EXTRACT3PIX1D extension because 3 pixels have better S/N
        hdr = hdu[2].header # this is the header
        wave = tab["WAVELENGTH"]/(1+z) # this is in microns
        flux = tab["FLUX"]*1e20 # erg s-1 cm-2 AA-1
        flux_err = tab["FLUX_ERR"]*1e20
    
    if row["O3_5007_combined_flux"]:
        norm = 1
    else:
        norm = 1
                
    flux_rs, flux_err_rs = spectres(wave_common_weird_sources, wave, flux, spec_errs=flux_err, fill=np.nan)
    flux_norm = flux_rs/norm 
    flux_norm_err = flux_err_rs/norm

    # plot 1:

    fig, ax = plt.subplots(figsize=(12, 3), dpi=100)
    plt.subplots_adjust(top=0.80)  # creates label band
    
    ax.plot(wave_common_weird_sources, flux_norm, color=blue, lw=1)
    ax.fill_between(wave_common_weird_sources, flux_norm - flux_norm_err, flux_norm + flux_norm_err, color=blue, alpha=0.5, step="mid",)
    ax.set_xlabel(r"Rest Wavelength ($\mu$m)")
    ax.set_ylabel(r"Flux$_\lambda$ ($10^{-20}$ erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$)")
    
    label_emission_lines(ax, emission_lines, y_frac=1.05)
    
    ax.set_title(f"Type 1 Prism Source: {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)
    
    plt.tight_layout()
    plt.show()

    # plot 2:

    fig, ax = plt.subplots(figsize=(4, 3), dpi=200)
    plt.subplots_adjust(top=0.80)  # creates label band
    
    ax.plot(wave_common_weird_sources, flux_norm, color=pink, lw=1)
    ax.fill_between(wave_common_weird_sources, flux_norm - flux_norm_err, flux_norm + flux_norm_err, color=pink, alpha=0.5, step="mid",)
    ax.set_xlabel(r"Rest Wavelength ($\mu$m)")
    ax.set_ylabel(r"Flux$_\lambda$ ($10^{-20}$ erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$)")
    
    label_emission_lines(ax, optical_emission_lines, y_frac=1.05)
    
    ax.set_title( f"Type 1 Prism Source: {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)

    plt.xlim(0.14, 0.2)
    
    plt.tight_layout()
    plt.show()



## Emission Lines

In [ ]:
emission_line_table = table.Table.read("JADES Emission Lines/new_lines_table_edit.csv", delimiter = ",")
emission_line_table["index"] = np.arange(0, len(emission_line_table))
emission_line_table.write("JADES Emission Lines/new_lines_table_edit_urvi.csv",format="csv", overwrite=True)

emission_line_waves = emission_line_table["Wavelength"] * 1e-4 # convert A to um
emission_lines = {
    r"Ly$\alpha$": emission_line_waves[44], # 0.1216,    
    # r"NV": emission_line_waves[49] # 0.1239,
    # r"SiIV": emission_line_waves[61] # 0.1403,
    r"NIV]": emission_line_waves[63], # 0.1483,
    r"CIV": emission_line_waves[68], # 0.1651,
    # r"HeII": emission_line_waves[71] # 0.16404,
    r"OIII]": emission_line_waves[74], # 0.166585,
    r"NIII]": emission_line_waves[80], # 0.1750,
    r"CIII]": emission_line_waves[93], # 0.1908734,
    # r"[NeV]": emission_line_waves[94] # 0.2141,
    # r"[NeIV]": emission_line_waves[107] # 0.24395,
    r"MgII": emission_line_waves[114], # 0.2796,

    r"[NeV]": emission_line_waves[121], # 0.3346,
    r"[OII]": emission_line_waves[127], # 0.3728,
    r"[NeIII]": emission_line_waves[132], # 0.386876,
    r"[HeI]": emission_line_waves[133], # 0.3889,
    r"H$\epsilon$": emission_line_waves[138], # 0.39701,
    r"H$\delta$": emission_line_waves[143], # 0.410289,
    r"H$\gamma$": emission_line_waves[146], # 0.434168,
    r"[OIII] aurora": emission_line_waves[147], # 0.4364436,
    
    r"H$\beta$": emission_line_waves[157], # 0.4861,
    r"[OIII]": emission_line_waves[160], # 0.4959,
    r"[OIII]": emission_line_waves[163], # 0.5007,

    r"[OI]": emission_line_waves[181], # 0.6302046,
    r"[OI]": emission_line_waves[183], # 0.6365536, 
    r"[NII]": emission_line_waves[185], #
    r"H$\alpha$": emission_line_waves[186], # 0.6563,
    r"[NII] doub": emission_line_waves[187], #
    # r"[NII]": emission_line_waves[187] # 0.6583,
    r"[SII]": emission_line_waves[190], # 0.671729,
    r"[SII]": emission_line_waves[191] # 0.673267,
}

def label_emission_lines(ax, lines, y_frac=1.05):
    """
    Place emission-line labels above the axes, outside the plot region.
    
    ax      : matplotlib axis
    lines   : dict {label: wavelength}
    y_frac  : axes fraction (>1 places text outside)
    """
    for label, wl in lines.items():
        # vertical guide line inside plot
        ax.axvline(wl, color="0.6", lw=0.8, ls="--", zorder=0)

        # label outside
        ax.text(wl, y_frac, label, rotation=90, ha="center", va="bottom", fontsize=9, color="0.3", 
                transform=ax.get_xaxis_transform(), clip_on=False)


## Fitting Functions

In [ ]:
# ----------------------------------------------------
# LSF-aware Gaussian (integrated flux parametrisation)
# ----------------------------------------------------

# def sigma_lsf(lam_um, z_med):
    
#     lam_obs = lam_um*(1+z_med)
#     R = R_interp(lam_obs) # R can be only calculated at observed wave
#     FWHM_obs = lam_obs / R
#     FWHM_rest = FWHM_obs / (1 + z_med)
#     return FWHM_rest / 2.355

# def gauss_lsf_flux(lam, lam0, F):

#     sigma = sigma_lsf(lam0, z_med) # scalar σ at line center
#     amp = F / (sigma * np.sqrt(2*np.pi))
#     return amp * np.exp(-0.5 * ((lam - lam0) / sigma)**2)

def sigma_lsf(lam_um, z_med):

    lam_obs = lam_um * (1 + z_med)
    R = R_interp(lam_obs)

    FWHM_obs = lam_obs / R
    FWHM_rest = FWHM_obs / (1 + z_med)

    sigma_inst = FWHM_rest / 2.355
    return sigma_inst

def gauss_lsf_flux(lam, lam0, F):
    sigma = sigma_lsf(lam0, z_med)
    amp = F / (sigma * np.sqrt(2*np.pi))
    return amp * np.exp(-0.5 * ((lam - lam0) / sigma)**2)
    
# ----------------------------------------------------
# Balmer Break Added in the continuum:
# ----------------------------------------------------

def balmer_continuum(lam, A_blue, alpha_blue, A_red, alpha_red): # Balmer-break continuum fit

    lam = np.asarray(lam)
    lam_blue = 0.3640 
    lam_red = 0.3820
    lam0 = 0.37 # pivot point helps in normalization 

    f = np.zeros_like(lam)

    # ----- blue side ------:
    m_blue = lam < lam_blue
    f[m_blue] = A_blue * ((lam[m_blue]/lam0)**alpha_blue)

    # ----- red side ------:
    m_red = lam > lam_red
    f[m_red] = A_red * ((lam[m_red]/lam0)**alpha_red)

    # ------ mid ------:
    m_mid = (lam >= lam_blue) & (lam <= lam_red)

    if np.any(m_mid):
        
        f_blue_edge = A_blue * ((lam_blue/lam0)**alpha_blue)
        f_red_edge = A_red * ((lam_red/lam0)**alpha_red)
    
        f[m_mid] = (f_blue_edge + (f_red_edge-f_blue_edge) * ((lam[m_mid]-lam_blue)/(lam_red-lam_blue)))
        
    return f

# ----------------------------------------------------
# Masking Emission Lines
# ----------------------------------------------------
    
def line_mask(lam, line_waves, nsig=3):
    mask = np.ones_like(lam, dtype=bool)
    for lw in line_waves:
        sig = sigma_lsf(lw)
        mask &= ~((lam > lw - nsig*sig) & (lam < lw + nsig*sig))
    return mask

def shade_line_masks(ax, line_waves, nsig=3, color="gray", alpha=0.15):
    for lw in line_waves:
        sig = sigma_lsf(lw)
        ax.axvspan(lw - nsig*sig, lw + nsig*sig, color=color, alpha=alpha, lw=0)

optical_lines = [
    emission_line_waves[121], # 0.3346,
    emission_line_waves[127], # 0.3728,
    emission_line_waves[132], # 0.386876,
    emission_line_waves[133], # 0.3889,
    emission_line_waves[138], # 0.39701,
    emission_line_waves[143], # 0.410289,
    emission_line_waves[146], # 0.434168,
    emission_line_waves[147], # 0.4364436,
    emission_line_waves[157], # 0.4861,
    emission_line_waves[160], # 0.4959,
    emission_line_waves[163], # 0.5007,

    # emission_line_waves[181], # 0.6302046,
    # emission_line_waves[183], # 0.6365536, 
    emission_line_waves[185],
    emission_line_waves[186],
    emission_line_waves[187] 
]

# ------- UV region emission lines ----- :

optical_emission_lines = {
    
    r"[NeV]": emission_line_waves[121], # 0.3346,
    r"[OII]": emission_line_waves[127], # 0.3728,
    r"[NeIII]": emission_line_waves[132], # 0.386876,
    r"[HeI]": emission_line_waves[133], # 0.3889,
    r"H$\epsilon$": emission_line_waves[138], # 0.39701,
    r"H$\delta$": emission_line_waves[143], # 0.410289,
    r"H$\gamma$": emission_line_waves[146], # 0.434168,
    r"[OIII] aurora": emission_line_waves[147], # 0.4364436,
    
    r"H$\beta$": emission_line_waves[157], # 0.4861,
    r"[OIII]": emission_line_waves[160], # 0.4959,
    r"[OIII]": emission_line_waves[163], # 0.5007,

    # r"[OI]": emission_line_waves[181], # 0.6302046,
    # r"[OI]": emission_line_waves[183], # 0.6365536,  
    r"[NII]": emission_line_waves[185], #
    r"H$\alpha$": emission_line_waves[186], # 0.6563,
    r"[NII] doub": emission_line_waves[187]
}

def opt_power_law(lam, A_blue, alpha_blue, A_red, alpha_red, lam0=0.5):
    return A_red * (lam / lam0)**alpha_red

cont_bounds = (
    [0.0, -5.0, 0.0, -5.0],   # lower
    [np.inf, 0.0, np.inf, 0.0]  # upper
)


# ----------------------------------------------------
# UV + optical emission-line wavelengths (micron)
# ----------------------------------------------------

# --- Optical emission line wavelengths ---

LAM_NeV        = emission_line_waves[121]  # 0.3346
LAM_OII        = emission_line_waves[127]  # 0.3728
LAM_NeIII      = emission_line_waves[132]  # 0.386876
LAM_HeI        = emission_line_waves[133]  # 0.3889
LAM_Hep        = emission_line_waves[138]  # H epsilon 0.39701
LAM_Hd         = emission_line_waves[143]  # H delta 0.410289
LAM_Hg         = emission_line_waves[146]  # H gamma 0.434168
LAM_OIII_aur   = emission_line_waves[147]  # 0.4364436

LAM_Hb         = emission_line_waves[157]  # 0.4861
LAM_OIII_1     = emission_line_waves[160]  # 0.4959
LAM_OIII_2     = emission_line_waves[163]  # 0.5007

# Optional [OI] doublet if needed later
# LAM_OI_1    = emission_line_waves[181]  # 0.6302046
# LAM_OI_2    = emission_line_waves[183]  # 0.6365536
LAM_NII_1      = emission_line_waves[185] 
LAM_Ha         = emission_line_waves[186]  # 0.6563
LAM_NII_2      = emission_line_waves[187] 

# ----------------------------------------------------
# Optical stacked-spectrum model
# ----------------------------------------------------

# def opt_power_law(lam, A, alpha, lam0=0.18):
#     return A * (lam / lam0)**alpha



def optical_region_cont_fit_stack(lam,
                                  F_NeV, F_OII, F_NeIII, F_HeI,F_Heps, F_Hd, F_Hg, F_OIII_aur,F_Hb, F_OIII_1, F_OIII_2,F_Ha,F_NII,
                                  A_blue, alpha_blue, A_red, alpha_red):

    m = np.zeros_like(lam)
    cont = opt_power_law(lam, A_blue, alpha_blue, A_red, alpha_red)

    m += gauss_lsf_flux(lam, LAM_NeV, F_NeV)
    m += gauss_lsf_flux(lam, LAM_OII, F_OII)
    m += gauss_lsf_flux(lam, LAM_NeIII, F_NeIII)
    m += gauss_lsf_flux(lam, LAM_HeI, F_HeI)

    m += gauss_lsf_flux(lam, LAM_Hep, F_Heps)
    m += gauss_lsf_flux(lam, LAM_Hd, F_Hd)
    m += gauss_lsf_flux(lam, LAM_Hg, F_Hg)
    m += gauss_lsf_flux(lam, LAM_OIII_aur, F_OIII_aur)

    m += gauss_lsf_flux(lam, LAM_Hb, F_Hb)
    m += gauss_lsf_flux(lam, LAM_OIII_1, F_OIII_1)
    m += gauss_lsf_flux(lam, LAM_OIII_2, F_OIII_2)

    m += gauss_lsf_flux(lam, LAM_Ha, F_Ha)
    m += gauss_lsf_flux(lam, LAM_NII_1, F_NII/4.0)
    m += gauss_lsf_flux(lam, LAM_NII_2, 3*F_NII/4.0)
    
    return cont + m

## Resampling and Normalization

In [ ]:
# -------- resampling -------- :
mean_lsf = stellar_prop = table.Table.read("Line Spread Functions/meanlsf_clear_prism.csv", delimiter=",")
mean_lsf

wave_R = mean_lsf["wave"]*1e-4     # wave in microns
R = mean_lsf["R"]        # resolving power

# interpolator for R(lambda), # the fill value makes the R always have true values 
R_interp = interp1d(wave_R, R, kind="linear", bounds_error=False, fill_value = (R[0], R[-1])) 

# Build a rest-frame wavelength grid matched to PRISM resolution at median redshift z_med.

def make_prism_rest_wave_grid(lam_min_rest,lam_max_rest,z_med,R_interp):

    wave_rest = [lam_min_rest]

    while wave_rest[-1] < lam_max_rest: # looping through all wavelengths 
        lam0_rest = wave_rest[-1]

        # observed-frame wavelength
        lam0_obs = lam0_rest * (1.0 + z_med)

        # resolving power at observed wavelength
        R0 = R_interp(lam0_obs)

        # observed-frame FWHM
        FWHM_obs = lam0_obs / R0

        # de-redshift to rest-frame
        FWHM_rest = FWHM_obs / (1.0 + z_med)

        # Nyquist step
        dlam_rest = FWHM_rest / 2.0

        wave_rest.append(lam0_rest + dlam_rest)

    return np.array(wave_rest)

# --------- normalization (using continuum) --------- :

cont_min = 0.20  # microns (2050 Å)
cont_max = 0.22  # microns (2150 Å)
pivot = 0.21

def continuum_norm(wave, flux, cont_min=0.2, cont_max=0.22, pivot=0.21):
    """
    Estimate continuum normalization using a linear fit.

    Parameters
    ----------
    wave : array
        Rest-frame wavelength (microns)
    flux : array
        Flux array
    cont_min, cont_max : float
        Continuum window limits (microns)
    pivot : float
        Wavelength where continuum is evaluated (microns)

    Returns
    -------
    norm : float
        Continuum normalization value
    """

    # continuum region
    mask = (wave > cont_min) & (wave < cont_max)

    wave_cont = wave[mask]
    flux_cont = flux[mask]

    # remove NaNs
    valid = np.isfinite(wave_cont) & np.isfinite(flux_cont)
    wave_cont = wave_cont[valid]
    flux_cont = flux_cont[valid]

    if len(wave_cont) > 2:

        # linear fit
        slope, intercept = np.polyfit(wave_cont, flux_cont, 1)

        norm = slope * pivot + intercept

    else:
        norm = np.nan

    # enforce positive normalization
    if not np.isfinite(norm) or norm <= 0:
        norm = np.nanmedian(np.abs(flux_cont))

    if not np.isfinite(norm) or norm == 0:
        norm = 1.0

    return norm
    


# ----- resolution at a specific wave -----

# z_med = np.nanmedian(compact["z_Spec_1"]) # depends on the group you choose
# lam_Ha_rest = 0.6563 # microns
# lam_Ha_obs = lam_Ha_rest * (1 + z_med) # observed wavelength
# R_Ha = R_interp(lam_Ha_obs) # resolving power at that wavelength
# FWHM_obs = lam_Ha_obs / R_Ha # observed-frame FWHM
# FWHM_rest = FWHM_obs / (1 + z_med) # rest-frame FWHM
# dlam_rest = FWHM_rest / 2 # Nyquist sampling step (same as your grid spacing)

# print("Hα observed wavelength:", lam_Ha_obs, "micron")
# print(f"R at Hα: {R_Ha:3f}")
# print(f"FWHM_obs: {FWHM_obs:.3f} micron")
# print(f"FWHM_rest: {FWHM_rest:3f} micron")
# print(f"Grid step: {dlam_rest:3f} micron")
mean_lsf

In [ ]:
z_med = np.nanmedian(compact["z_Spec_1"])

# --- rest-frame wave grid ---
wave_common_compact = make_prism_rest_wave_grid(lam_min_rest=0.1, lam_max_rest=0.8, z_med=z_med, R_interp=R_interp)

for i,row in enumerate(compact[:2]):

    print(row["file_prism"])
    z = row["z_Spec_1"]
    with fits.open(row["file_prism"]) as hdu:
        # hdu.info()
        tab = Table(hdu[2].data) # using the EXTRACT3PIX1D extension because 3 pixels have better S/N
        hdr = hdu[2].header # this is the header
        wave = tab["WAVELENGTH"]/(1+z) # this is in microns
        flux = tab["FLUX"]*1e20 # erg s-1 cm-2 AA-1
        flux_err = tab["FLUX_ERR"]*1e20

    if row["O3_5007_combined_flux"]:
        norm = 1
    else:
        norm = 1

    # --- normalization ---
    norm = continuum_norm(wave, flux)
        
    flux_rs, flux_err_rs = spectres(wave_common_compact, wave, flux, spec_errs=flux_err, fill=np.nan)
    
    flux_norm = flux_rs/norm 
    flux_norm_err = flux_err_rs/norm

    
    # plot 1:

    fig, ax = plt.subplots(figsize=(12, 3), dpi=100)
    plt.subplots_adjust(top=0.80)  # creates label band
    
    ax.plot(wave_common_compact, flux_norm, color=blue, lw=1)
    ax.fill_between(wave_common_compact, flux_norm - flux_norm_err, flux_norm + flux_norm_err, color=blue, alpha=0.5, step="mid",)
    ax.set_xlabel(r"Rest Wavelength ($\mu$m)")
    ax.set_ylabel(r"Flux$_\lambda$ ($10^{-20}$ erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$)")
    
    label_emission_lines(ax, emission_lines, y_frac=1.05)
    
    ax.set_title(f"Type 1 Prism Source: {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)
    
    plt.tight_layout()
    plt.show()

# Bad Sources Check

In [ ]:
bad_sources = []

for i, row in enumerate(intermediate[:]):
    # print(i+1)
    z = row["z_Spec_1"]

    with fits.open(row["file_prism"]) as hdu:
        tab = Table(hdu[2].data)
        wave = tab["WAVELENGTH"]/(1+z)
        flux = tab["FLUX"]*1e20
        flux_err = tab["FLUX_ERR"]*1e20

    norm = continuum_norm(wave, flux)
    # norm_compact.append(norm)

    flux_rs, flux_err_rs = spectres(
        wave_common_compact, wave, flux,
        spec_errs=flux_err, fill=np.nan
    )

    flux_norm = flux_rs / norm
    flux_norm_err = flux_err_rs / norm

    wave = wave_common_compact
    flux = flux_norm
    err = flux_norm_err

    mask = (wave > 0.2) & (wave < 0.7)

    lam = wave[mask]
    f = flux[mask]
    e = err[mask]

    # ---- check for NaN / inf ----
    if not (np.all(np.isfinite(f)) and np.all(np.isfinite(e))):
        bad_sources.append(row["ID"])
        print("NaN/Inf detected in:", row["ID"])

print(bad_sources)

# Compact Sources

In [ ]:
compact

## Median Stack for Compact Sources

In [ ]:
z_med_compact = np.nanmedian(compact["z_Spec_1"])
wave_common_compact = make_prism_rest_wave_grid(lam_min_rest=0.1, lam_max_rest=0.8, z_med=z_med_compact, R_interp=R_interp)

opt_min, opt_max = 0.3, 0.7

mask_opt = (wave_common_compact > opt_min) & (wave_common_compact < opt_max)
wave_opt_compact = wave_common_compact[mask_opt]

f_compact = []
err_compact = []
norm_compact = []

for i,row in enumerate(compact[:]):
    print(i+1)
    z = row["z_Spec_1"]
    with fits.open(row["file_prism"]) as hdu:

        tab = Table(hdu[2].data) # using the EXTRACT3PIX1D extension because 3 pixels have better S/N
        wave = tab["WAVELENGTH"]/(1+z) # this is in microns
        flux = tab["FLUX"]*1e20 # erg s-1 cm-2 AA-1
        flux_err = tab["FLUX_ERR"]*1e20
    
    if row["O3_5007_combined_flux"]:
        norm = 1
    else:
        norm = 1

    norm = continuum_norm(wave, flux)   
    norm_compact.append(norm)
    flux_rs, flux_err_rs = spectres(wave_common_compact, wave, flux, spec_errs=flux_err, fill=np.nan)
    flux_norm = flux_rs/norm 
    flux_norm_err = flux_err_rs/norm
    
    wave = wave_common_compact
    flux = flux_norm 
    err = flux_norm_err 
    mask = ((wave > 0.3) & (wave < 0.7))# & np.isfinite(flux) & np.isfinite(err)) 
    
    lam = wave[mask] 
    f = flux[mask] 
    e = err[mask]
    print(row["ID"])
        
    fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.5),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})
    
    # -------------------
    # Top: spectrum + fit
    # -------------------
    
    # observed spectrum
    ax.step(lam, f, where="mid", lw=1, color=blue, label="Spectrum")
    ax.fill_between(lam, f - e, f + e, color=blue, alpha=0.3)
    
    # continuum model
    # ax.plot(lam,cont_model,color="k",lw=1.2,alpha=0.9,label="Continuum fit")
    
    # points actually used in the fit
    # ax.scatter(lam[mask],f[mask],s=8,color="k",zorder=5,label="Fit region")
    
    # plotting the continuum subtracted spectra
    ax.step(lam,f,where="mid",lw=1,color="#B34280",label="Continuum-subtracted")
    ax.fill_between(lam,f - e,f + e,alpha=0.3, color = "#B34280")
    ax.axhline(0.0, color="gray", lw=0.8)

    label_emission_lines(ax, optical_emission_lines, y_frac=1.05)
    ax.set_ylabel(r"Normalized flux$_\lambda$")
    ax.legend(frameon=False, fontsize=8)
    
    # -------------------
    # Bottom: residuals (continuum subtracted)
    # -------------------
    
    axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
    axr.axhline(1.0, color="gray", ls="--", lw=0.6)
    axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
    
    axr.step(lam,f / e,where="mid",color="#9E3C55",lw=0.8)
    
    axr.set_ylabel(r"$(f - c)/\sigma$")
    axr.set_xlabel(r"Rest wavelength ($\mu$m)")
    axr.set_ylim(-8, 8)


    ax.set_title(f"Continuum Fits for Compact Prism Source {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)

    
    plt.tight_layout()
    plt.show()

    # ------- saving the continuum subtarcted flux ------- :
    
    # Create OPT-only arrays
    f_opt = np.full_like(wave_opt_compact, np.nan)
    e_opt = np.full_like(wave_opt_compact, np.nan)
    
    # Map lam → wave_opt_compact (they are identical grids if you used spectres earlier)
    f_opt[:] = f
    e_opt[:] = e
    
    f_compact.append(f_opt)
    err_compact.append(e_opt)

In [ ]:
print(min(norm_compact))

In [ ]:
# ----- median stacking ------:
# %matplotlib widget
opt_min, opt_max = 0.3, 0.7
mask_opt = (wave_common_compact > opt_min) & (wave_common_compact < opt_max)
wave_opt_compact = wave_common_compact[mask_opt]

f_compact = np.array(f_compact)   # (Nspec, Nopt)
err_compact = np.array(err_compact)

median_stacked_flux_opt_compact = np.nanmedian(f_compact, axis=0)

rng = np.random.default_rng()
n_mc = 1000

stacked_mc = np.empty((n_mc, len(wave_opt_compact)))

for k in range(n_mc):
    flux_pert = rng.normal(f_compact, err_compact)
    stacked_mc[k] = np.nanmedian(flux_pert, axis=0)

median_stacked_err_opt_compact = np.nanstd(stacked_mc, axis=0)
median_stacked_n_opt_compact = np.sum(~np.isnan(f_compact), axis=0)

median_stacked_flux_opt_compact = median_stacked_flux_opt_compact
median_stacked_err_opt_compact = median_stacked_err_opt_compact

median_stacked_resid_opt_compact = median_stacked_flux_opt_compact / median_stacked_err_opt_compact


fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.8),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# -------------------
# Top: opt stack
# -------------------

ax.step(wave_opt_compact,median_stacked_flux_opt_compact,where="mid",lw=1.2,label="Median stack (cont-sub)", color = "#42A2B3")

ax.fill_between(wave_opt_compact,median_stacked_flux_opt_compact - median_stacked_err_opt_compact,median_stacked_flux_opt_compact + median_stacked_err_opt_compact,alpha=0.3, color = "#42A2B3")

ax.axhline(0.0, color="gray", lw=0.8)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.set_ylabel(r"Normalized flux")
ax.legend(frameon=False, fontsize=8)

# -------------------
# Bottom: residuals
# -------------------

axr.step(wave_opt_compact,median_stacked_resid_opt_compact,where="mid",lw=0.9,color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")
axr.set_ylim(-8, 8)

# ax.set_title("opt continuum-subtracted compact median stack", pad =35)
ax.set_title("opt continuum compact median stack", pad =35)
plt.tight_layout()
plt.show()


# ------- fitting the median stack ------ :

lam = wave_opt_compact
f   = median_stacked_flux_opt_compact
e   = median_stacked_err_opt_compact

# ------- line model ------- :

p0 = [
    1,1,1,1,1,1,1,1,1,1,1,1,1,      # line fluxes
    np.nanmedian(f), -2.0,        # continuum blue
    np.nanmedian(f), -1.0]        # continuum red                       

bounds = (
    [0,0,0,0,0,0,0,0,0,0,0,0,0, 0,-np.inf,0,-np.inf],
    [np.inf,np.inf,np.inf,np.inf,np.inf,np.inf,np.inf,np.inf,
     np.inf,np.inf,np.inf,np.inf,np.inf, np.inf,np.inf,np.inf,np.inf]
)


popt_median_compact, pcov_median_compact = curve_fit(
    optical_region_cont_fit_stack,
    lam, f,
    sigma=e,
    p0=p0,
    bounds=bounds,
    absolute_sigma=True
)

perr_median_compact = np.sqrt(np.diag(pcov_median_compact))

model_opt_median_compact = optical_region_cont_fit_stack(wave_opt_compact, *popt_median_compact)

# ------ continuum model ------ :

A_blue = popt_median_compact[-4]
alpha_blue = popt_median_compact[-3]

A_red = popt_median_compact[-2]
alpha_red = popt_median_compact[-1]

model_cont_median_compact = opt_power_law(wave_opt_compact, A_blue, alpha_blue, A_red, alpha_red)

# Individual components (for plotting)
m_nev = gauss_lsf_flux(wave_opt_compact, LAM_NeV, popt_median_compact[0])
m_oii = gauss_lsf_flux(wave_opt_compact, LAM_OII, popt_median_compact[1])
m_neiii = gauss_lsf_flux(wave_opt_compact, LAM_NeIII, popt_median_compact[2])
m_hei = gauss_lsf_flux(wave_opt_compact, LAM_HeI, popt_median_compact[3])

m_heps = gauss_lsf_flux(wave_opt_compact, LAM_Hep, popt_median_compact[4])
m_hd = gauss_lsf_flux(wave_opt_compact, LAM_Hd, popt_median_compact[5])
m_hg = gauss_lsf_flux(wave_opt_compact, LAM_Hg, popt_median_compact[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_compact, LAM_OIII_aur, popt_median_compact[7])

m_hb = gauss_lsf_flux(wave_opt_compact, LAM_Hb, popt_median_compact[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_compact, LAM_OIII_1, popt_median_compact[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_compact, LAM_OIII_2, popt_median_compact[10])

m_ha = gauss_lsf_flux(wave_opt_compact, LAM_Ha, popt_median_compact[11])
m_nii_1 = gauss_lsf_flux(wave_opt_compact,LAM_NII_1,popt_median_compact[12])
m_nii_2 = gauss_lsf_flux(wave_opt_compact,LAM_NII_2,3*popt_median_compact[12])


# ----------------------------------------------------
# Plot: spectrum + residuals
# ----------------------------------------------------

fig, (ax, axr) = plt.subplots(2, 1,figsize=(10, 3),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_opt_compact,median_stacked_flux_opt_compact,where="mid",lw=1.2,color="#42A2B3",label="Median stack")

ax.fill_between(wave_opt_compact,median_stacked_flux_opt_compact - median_stacked_err_opt_compact,median_stacked_flux_opt_compact + median_stacked_err_opt_compact,alpha=0.3,color="#42A2B3")

ax.plot(wave_opt_compact, model_opt_median_compact, color="k", lw=1.2, label="Optical Model Fit")
ax.plot(wave_opt_compact, model_cont_median_compact, color="k", lw=1.2, linestyle = "dashed")

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.plot(wave_opt_compact, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_median_compact[0]/perr_median_compact[0]:.2f}")
ax.plot(wave_opt_compact, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_median_compact[1]/perr_median_compact[1]:.2f}")
ax.plot(wave_opt_compact, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_median_compact[2]/perr_median_compact[2]:.2f}")
ax.plot(wave_opt_compact, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_median_compact[3]/perr_median_compact[3]:.2f}")

ax.plot(wave_opt_compact, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_median_compact[4]/perr_median_compact[4]:.2f}")
ax.plot(wave_opt_compact, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_median_compact[5]/perr_median_compact[5]:.2f}")
ax.plot(wave_opt_compact, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_median_compact[6]/perr_median_compact[6]:.2f}")
ax.plot(wave_opt_compact, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_median_compact[7]/perr_median_compact[7]:.2f}")

ax.plot(wave_opt_compact, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_median_compact[8]/perr_median_compact[8]:.2f}")
ax.plot(wave_opt_compact, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_median_compact[9]/perr_median_compact[9]:.2f}")
ax.plot(wave_opt_compact, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_median_compact[10]/perr_median_compact[10]:.2f}")
ax.plot(wave_opt_compact, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_median_compact[11]/perr_median_compact[11]:.2f}")

ax.plot(wave_opt_compact, m_nii_1, ls="--", lw=0.9, label="[NII]6548")
ax.plot(wave_opt_compact, m_nii_2, ls="--", lw=0.9, label="[NII]6583")

ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Normalized Flux")

# ax.legend(frameon=False,loc="center left",bbox_to_anchor=(1.02, 0.25),ncol=1)
ax.set_title("Optical Compact Sources Median stack", pad=30)

# ---- bottom panel ----
resid = median_stacked_flux_opt_compact - model_opt_median_compact
axr.step(wave_opt_compact, resid / median_stacked_err_opt_compact, where="mid", lw=0.9, color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

plt.tight_layout(rect=[0, 0, 0.83, 1])
plt.show()

# ------  checking the significane of the line detections -------- :
# print(f"S/N  of Lya] = {popt_median_compact[1]/perr_median_compact[1]}")
# print(f"S/N  of NIV] = {popt_median_compact[2]/perr_median_compact[2]}")
# print(f"S/N  of CIV = {popt_median_compact[3]/perr_median_compact[3]}")
# print(f"S/N  of HeII = {popt_median_compact[4]/perr_median_compact[4]}")
# print(f"S/N  of OIII] = {popt_median_compact[5]/perr_median_compact[5]}")
# print(f"S/N  of NIII] = {popt_median_compact[6]/perr_median_compact[6]}")
# print(f"S/N  of CIII] = {popt_median_compact[7]/perr_median_compact[7]}")


## Mean Stack for Compact Sources

In [ ]:
%matplotlib widget
z_med_compact = np.nanmedian(compact["z_Spec_1"])
wave_common_compact = make_prism_rest_wave_grid(lam_min_rest=0.1, lam_max_rest=0.8, z_med=z_med_compact, R_interp=R_interp)

opt_min, opt_max = 0.3, 0.7

mask_opt = (wave_common_compact > opt_min) & (wave_common_compact < opt_max)
wave_opt_compact = wave_common_compact[mask_opt]

f_compact = []
err_compact = []

for i,row in enumerate(compact[:]):
    print(i+1)
    z = row["z_Spec_1"]
    with fits.open(row["file_prism"]) as hdu:

        tab = Table(hdu[2].data) # using the EXTRACT3PIX1D extension because 3 pixels have better S/N
        wave = tab["WAVELENGTH"]/(1+z) # this is in microns
        flux = tab["FLUX"]*1e20 # erg s-1 cm-2 AA-1
        flux_err = tab["FLUX_ERR"]*1e20
    
    if row["O3_5007_combined_flux"]:
        norm = 1
    else:
        norm = 1

    norm = continuum_norm(wave, flux)
    flux_rs, flux_err_rs = spectres(wave_common_compact, wave, flux, spec_errs=flux_err, fill=np.nan)
    flux_norm = flux_rs/norm 
    flux_norm_err = flux_err_rs/norm
    
    wave = wave_common_compact
    flux = flux_norm 
    err = flux_norm_err 
    mask = ((wave > 0.3) & (wave < 0.7))# & np.isfinite(flux) & np.isfinite(err)) 
    
    lam = wave[mask] 
    f = flux[mask] 
    e = err[mask]
    print(row["ID"])
    
    # ----- sigma clipping ------- :
    clip_region = np.isfinite(f) & np.isfinite(e)
    clip_mask = np.zeros_like(f, dtype=bool)

    if np.any(clip_region):
        clipped = sigma_clip(
            f[clip_region],
            sigma=5.0,
            maxiters=5, 
            cenfunc="median",
            stdfunc=mad_std,
            masked=True
        ).mask
        clip_mask[clip_region] = np.asarray(clipped)

    f_clipped = f.copy()
    e_clipped = e.copy()
    f_clipped[clip_mask] = np.nan
    e_clipped[clip_mask] = np.nan

    # f = f_clipped - cont_model
    f = f_clipped
    e = e_clipped
    
    fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.5),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})
    
    # -------------------
    # Top: spectrum + fit
    # -------------------
    
    # observed spectrum
    ax.step(lam, f, where="mid", lw=1, color=blue, label="Spectrum")
    ax.fill_between(lam, f - e, f + e, color=blue, alpha=0.3)
    
    # continuum model
    # ax.plot(lam,cont_model,color="k",lw=1.2,alpha=0.9,label="Continuum fit")
    
    # points actually used in the fit
    # ax.scatter(lam[mask],f[mask],s=8,color="k",zorder=5,label="Fit region")
    
    # plotting the continuum subtracted spectra
    ax.step(lam,f,where="mid",lw=1,color="#B34280",label="Continuum-subtracted")
    ax.fill_between(lam,f - e,f + e,alpha=0.3, color = "#B34280")
    ax.axhline(0.0, color="gray", lw=0.8)

    label_emission_lines(ax, optical_emission_lines, y_frac=1.05)
    ax.set_ylabel(r"Normalized flux$_\lambda$")
    ax.legend(frameon=False, fontsize=8)
    
    # -------------------
    # Bottom: residuals (continuum subtracted)
    # -------------------
    
    axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
    axr.axhline(1.0, color="gray", ls="--", lw=0.6)
    axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
    
    axr.step(lam,f / e,where="mid",color="#9E3C55",lw=0.8)
    
    axr.set_ylabel(r"$(f - c)/\sigma$")
    axr.set_xlabel(r"Rest wavelength ($\mu$m)")
    axr.set_ylim(-8, 8)


    ax.set_title(f"Continuum Fits for Compact Prism Source {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)

    
    plt.tight_layout()
    plt.show()

    # ------- saving the continuum subtarcted flux ------- :
    
    # Create opt-only arrays
    f_opt = np.full_like(wave_opt_compact, np.nan)
    e_opt = np.full_like(wave_opt_compact, np.nan)
    
    # Map lam → wave_opt_compact (they are identical grids if you used spectres earlier)
    f_opt[:] = f
    e_opt[:] = e
    
    f_compact.append(f_opt)
    err_compact.append(e_opt)
        
f_compact = np.array(f_compact)   # (Nspec, Nopt)
err_compact = np.array(err_compact)

# --- variance-weighted mean stack ---
# weights are inverse variance; avoid divide-by-zero
w = 1.0 / np.where(err_compact > 0, err_compact**2, np.nan)

# --- variance-weighted mean stack ---

w = 1.0 / np.where(err_compact > 0, err_compact**2, np.nan) # weights are inverse variance; avoid divide-by-zero
w_norm = w / np.nansum(w, axis=0) # weights should be normalized!

mean_stacked_flux_opt_compact = np.nansum(w_norm * f_compact, axis=0)
mean_stacked_err_opt_compact  = np.sqrt(1.0 / np.nansum(w, axis=0))

mean_stacked_flux_opt_compact = mean_stacked_flux_opt_compact
mean_stacked_err_opt_compact = mean_stacked_err_opt_compact

mean_stacked_resid_opt_compact = mean_stacked_flux_opt_compact / mean_stacked_err_opt_compact
mean_stacked_n_opt_compact = np.sum(np.isfinite(f_compact), axis=0)


fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.8),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# -------------------
# Top: opt stack
# -------------------

ax.step(wave_opt_compact,mean_stacked_flux_opt_compact,where="mid",lw=1.2,label="Mean stack", color = "#42A2B3")

ax.fill_between(wave_opt_compact,mean_stacked_flux_opt_compact - mean_stacked_err_opt_compact,mean_stacked_flux_opt_compact + mean_stacked_err_opt_compact,alpha=0.3, color = "#42A2B3")

ax.axhline(0.0, color="gray", lw=0.8)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.set_ylabel(r"Normalized flux")
ax.legend(frameon=False, fontsize=8)

# -------------------
# Bottom: residuals
# -------------------

axr.step(wave_opt_compact,mean_stacked_resid_opt_compact,where="mid",lw=0.9,color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")
axr.set_ylim(-8, 8)

# ax.set_title("opt continuum-subtracted compact mean stack", pad =35)
ax.set_title("Optical continuum compact Mean stack", pad =35)
plt.tight_layout()
plt.show()


# ------- fitting the mean stack ------ :

lam = wave_opt_compact
f   = mean_stacked_flux_opt_compact
e   = mean_stacked_err_opt_compact

# ------- line model ------- :

p0 = [
    0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,      # line fluxes
    np.nanmedian(f), -2.0,        # continuum blue
    np.nanmedian(f), -1.0]        # continuum red


bounds = (
    [0,0,0,0,0,0,0,0,0,0,0,0,0, 0,-np.inf,0,-np.inf],
    [np.inf,np.inf,np.inf,np.inf,np.inf,np.inf,np.inf,np.inf,
     np.inf,np.inf,np.inf,np.inf,np.inf, np.inf,np.inf,np.inf,np.inf]
)


popt_mean_compact, pcov_mean_compact = curve_fit(
    optical_region_cont_fit_stack,
    lam, f,
    sigma=e,
    p0=p0,
    bounds=bounds,
    absolute_sigma=True
)

perr_mean_compact = np.sqrt(np.diag(pcov_mean_compact))

model_opt_mean_compact = optical_region_cont_fit_stack(wave_opt_compact, *popt_mean_compact)

# ------ continuum model ------ :

A_blue = popt_mean_compact[-4]
alpha_blue = popt_mean_compact[-3]

A_red = popt_mean_compact[-2]
alpha_red = popt_mean_compact[-1]


model_cont_mean_compact = opt_power_law(wave_opt_compact, A_blue, alpha_blue, A_red, alpha_red)

# Individual components (for plotting)
m_nev = gauss_lsf_flux(wave_opt_compact, LAM_NeV, popt_mean_compact[0])
m_oii = gauss_lsf_flux(wave_opt_compact, LAM_OII, popt_mean_compact[1])
m_neiii = gauss_lsf_flux(wave_opt_compact, LAM_NeIII, popt_mean_compact[2])
m_hei = gauss_lsf_flux(wave_opt_compact, LAM_HeI, popt_mean_compact[3])

m_heps = gauss_lsf_flux(wave_opt_compact, LAM_Hep, popt_mean_compact[4])
m_hd = gauss_lsf_flux(wave_opt_compact, LAM_Hd, popt_mean_compact[5])
m_hg = gauss_lsf_flux(wave_opt_compact, LAM_Hg, popt_mean_compact[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_compact, LAM_OIII_aur, popt_mean_compact[7])

m_hb = gauss_lsf_flux(wave_opt_compact, LAM_Hb, popt_mean_compact[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_compact, LAM_OIII_1, popt_mean_compact[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_compact, LAM_OIII_2, popt_mean_compact[10])

m_ha = gauss_lsf_flux(wave_opt_compact, LAM_Ha, popt_mean_compact[11])
m_nii_1 = gauss_lsf_flux(wave_opt_compact,LAM_NII_1,popt_mean_compact[12])
m_nii_2 = gauss_lsf_flux(wave_opt_compact,LAM_NII_2,3*popt_mean_compact[12])


# ----------------------------------------------------
# Plot: spectrum + residuals
# ----------------------------------------------------

fig, (ax, axr) = plt.subplots(2, 1,figsize=(10, 3),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_opt_compact,mean_stacked_flux_opt_compact,where="mid",lw=1.2,color="#42A2B3",label="Mean stack")

ax.fill_between(wave_opt_compact,mean_stacked_flux_opt_compact - mean_stacked_err_opt_compact,mean_stacked_flux_opt_compact + mean_stacked_err_opt_compact,alpha=0.3,color="#42A2B3")

ax.plot(wave_opt_compact, model_opt_mean_compact, color="k", lw=1.2, label="Optical Model Fit")
ax.plot(wave_opt_compact, model_cont_mean_compact, color="k", lw=1.2, linestyle = "dashed")

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.plot(wave_opt_compact, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_mean_compact[0]/perr_mean_compact[0]:.2f}")
ax.plot(wave_opt_compact, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_mean_compact[1]/perr_mean_compact[1]:.2f}")
ax.plot(wave_opt_compact, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_mean_compact[2]/perr_mean_compact[2]:.2f}")
ax.plot(wave_opt_compact, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_mean_compact[3]/perr_mean_compact[3]:.2f}")

ax.plot(wave_opt_compact, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_mean_compact[4]/perr_mean_compact[4]:.2f}")
ax.plot(wave_opt_compact, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_mean_compact[5]/perr_mean_compact[5]:.2f}")
ax.plot(wave_opt_compact, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_mean_compact[6]/perr_mean_compact[6]:.2f}")
ax.plot(wave_opt_compact, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_mean_compact[7]/perr_mean_compact[7]:.2f}")

ax.plot(wave_opt_compact, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_mean_compact[8]/perr_mean_compact[8]:.2f}")
ax.plot(wave_opt_compact, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_mean_compact[9]/perr_mean_compact[9]:.2f}")
ax.plot(wave_opt_compact, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_mean_compact[10]/perr_mean_compact[10]:.2f}")
ax.plot(wave_opt_compact, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_mean_compact[11]/perr_mean_compact[11]:.2f}")

ax.plot(wave_opt_compact, m_nii_1, ls="--", lw=0.9, label="[NII]6548")
ax.plot(wave_opt_compact, m_nii_2, ls="--", lw=0.9, label="[NII]6583")

ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Normalized Flux")

# ax.legend(frameon=False,loc="center left",bbox_to_anchor=(1.02, 0.25),ncol=1)
ax.set_title("Optical Compact Sources Mean stack", pad=30)

# ---- bottom panel ----
resid = mean_stacked_flux_opt_compact - model_opt_mean_compact
axr.step(wave_opt_compact, resid / mean_stacked_err_opt_compact, where="mid", lw=0.9, color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

plt.tight_layout(rect=[0, 0, 0.83, 1])
plt.show()

# ------  checking the significane of the line detections -------- :
# print(f"S/N  of Lya] = {popt_mean_compact[1]/perr_mean_compact[1]}")
# print(f"S/N  of NIV] = {popt_mean_compact[2]/perr_mean_compact[2]}")
# print(f"S/N  of CIV = {popt_mean_compact[3]/perr_mean_compact[3]}")
# print(f"S/N  of HeII = {popt_mean_compact[4]/perr_mean_compact[4]}")
# print(f"S/N  of OIII] = {popt_mean_compact[5]/perr_mean_compact[5]}")
# print(f"S/N  of NIII] = {popt_mean_compact[6]/perr_mean_compact[6]}")\
# print(f"S/N  of CIII] = {popt_mean_compact[7]/perr_mean_compact[7]}")




# Non-Compact Sources

In [ ]:
extended

## Median Stack for Extended Sources

In [ ]:
z_med_extended = np.nanmedian(extended["z_Spec_1"])
wave_common_extended = make_prism_rest_wave_grid(lam_min_rest=0.1, lam_max_rest=0.8, z_med=z_med_extended, R_interp=R_interp)

opt_min, opt_max = 0.3, 0.7

mask_opt = (wave_common_extended > opt_min) & (wave_common_extended < opt_max)
wave_opt_extended = wave_common_extended[mask_opt]

f_extended = []
err_extended = []
norm_ext = []

for i,row in enumerate(extended[:]):
    print(i+1)
    z = row["z_Spec_1"]
    with fits.open(row["file_prism"]) as hdu:

        tab = Table(hdu[2].data) # using the EXTRACT3PIX1D extension because 3 pixels have better S/N
        wave = tab["WAVELENGTH"]/(1+z) # this is in microns
        flux = tab["FLUX"]*1e20 # erg s-1 cm-2 AA-1
        flux_err = tab["FLUX_ERR"]*1e20
    
    if row["O3_5007_combined_flux"]:
        norm = 1
    else:
        norm = 1

    norm = continuum_norm(wave, flux)   
    norm_extended.append(norm)
    flux_rs, flux_err_rs = spectres(wave_common_extended, wave, flux, spec_errs=flux_err, fill=np.nan)
    flux_norm = flux_rs/norm 
    flux_norm_err = flux_err_rs/norm
    
    wave = wave_common_extended
    flux = flux_norm 
    err = flux_norm_err 
    mask = ((wave > 0.3) & (wave < 0.7))# & np.isfinite(flux) & np.isfinite(err)) 
    
    lam = wave[mask] 
    f = flux[mask] 
    e = err[mask]
    print(row["ID"])
        
    fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.5),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})
    
    # -------------------
    # Top: spectrum + fit
    # -------------------
    
    # observed spectrum
    ax.step(lam, f, where="mid", lw=1, color=blue, label="Spectrum")
    ax.fill_between(lam, f - e, f + e, color=blue, alpha=0.3)
    
    # continuum model
    # ax.plot(lam,cont_model,color="k",lw=1.2,alpha=0.9,label="Continuum fit")
    
    # points actually used in the fit
    # ax.scatter(lam[mask],f[mask],s=8,color="k",zorder=5,label="Fit region")
    
    # plotting the continuum subtracted spectra
    ax.step(lam,f,where="mid",lw=1,color="#B34280",label="Continuum-subtracted")
    ax.fill_between(lam,f - e,f + e,alpha=0.3, color = "#B34280")
    ax.axhline(0.0, color="gray", lw=0.8)

    label_emission_lines(ax, optical_emission_lines, y_frac=1.05)
    ax.set_ylabel(r"Normalized flux$_\lambda$")
    ax.legend(frameon=False, fontsize=8)
    
    # -------------------
    # Bottom: residuals (continuum subtracted)
    # -------------------
    
    axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
    axr.axhline(1.0, color="gray", ls="--", lw=0.6)
    axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
    
    axr.step(lam,f / e,where="mid",color="#9E3C55",lw=0.8)
    
    axr.set_ylabel(r"$(f - c)/\sigma$")
    axr.set_xlabel(r"Rest wavelength ($\mu$m)")
    axr.set_ylim(-8, 8)


    ax.set_title(f"Continuum Fits for extended Prism Source {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)

    
    plt.tight_layout()
    plt.show()

    # ------- saving the continuum subtarcted flux ------- :
    
    # Create OPT-only arrays
    f_opt = np.full_like(wave_opt_extended, np.nan)
    e_opt = np.full_like(wave_opt_extended, np.nan)
    
    # Map lam → wave_opt_extended (they are identical grids if you used spectres earlier)
    f_opt[:] = f
    e_opt[:] = e
    
    f_extended.append(f_opt)
    err_extended.append(e_opt)

In [ ]:
# ----- median stacking ------:
# %matplotlib widget
opt_min, opt_max = 0.3, 0.7
mask_opt = (wave_common_extended > opt_min) & (wave_common_extended < opt_max)
wave_opt_extended = wave_common_extended[mask_opt]

f_extended = np.array(f_extended)   # (Nspec, Nopt)
err_extended = np.array(err_extended)

median_stacked_flux_opt_extended = np.nanmedian(f_extended, axis=0)

rng = np.random.default_rng()
n_mc = 1000

stacked_mc = np.empty((n_mc, len(wave_opt_extended)))

for k in range(n_mc):
    flux_pert = rng.normal(f_extended, err_extended)
    stacked_mc[k] = np.nanmedian(flux_pert, axis=0)

median_stacked_err_opt_extended = np.nanstd(stacked_mc, axis=0)
median_stacked_n_opt_extended = np.sum(~np.isnan(f_extended), axis=0)

median_stacked_flux_opt_extended = median_stacked_flux_opt_extended
median_stacked_err_opt_extended = median_stacked_err_opt_extended

median_stacked_resid_opt_extended = median_stacked_flux_opt_extended / median_stacked_err_opt_extended


fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.8),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# -------------------
# Top: opt stack
# -------------------

ax.step(wave_opt_extended,median_stacked_flux_opt_extended,where="mid",lw=1.2,label="Median stack (cont-sub)", color = "#42A2B3")

ax.fill_between(wave_opt_extended,median_stacked_flux_opt_extended - median_stacked_err_opt_extended,median_stacked_flux_opt_extended + median_stacked_err_opt_extended,alpha=0.3, color = "#42A2B3")

ax.axhline(0.0, color="gray", lw=0.8)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.set_ylabel(r"Normalized flux")
ax.legend(frameon=False, fontsize=8)

# -------------------
# Bottom: residuals
# -------------------

axr.step(wave_opt_extended,median_stacked_resid_opt_extended,where="mid",lw=0.9,color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")
axr.set_ylim(-8, 8)

# ax.set_title("opt continuum-subtracted extended median stack", pad =35)
ax.set_title("opt continuum extended median stack", pad =35)
plt.tight_layout()
plt.show()


# ------- fitting the median stack ------ :

lam = wave_opt_extended
f   = median_stacked_flux_opt_extended
e   = median_stacked_err_opt_extended

# ------- line model ------- :

p0 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, np.nanmedian(f), -2.0, np.nanmedian(f), -1.0]
bounds = ([0,0, 0,0,0,0, 0,0,0, 0,0,0, 0, -np.inf, 0, -np.inf], 
          [np.inf, np.inf, np.inf, np.inf, np.inf,np.inf, np.inf, np.inf, np.inf, np.inf, np.inf,np.inf, np.inf, np.inf, np.inf, np.inf])

print(z_med)

popt_median_extended, pcov_median_extended = curve_fit(
    optical_region_cont_fit_stack,
    lam, f,
    sigma=e,
    p0=p0,
    bounds=bounds,
    absolute_sigma=True
)

perr_median_extended = np.sqrt(np.diag(pcov_median_extended))

model_opt_median_extended = optical_region_cont_fit_stack(wave_opt_extended, *popt_median_extended)

# ------ continuum model ------ :

A_blue = popt_median_extended[-4]
alpha_blue = popt_median_extended[-3]

A_red = popt_median_extended[-2]
alpha_red = popt_median_extended[-1]

model_cont_median_extended = opt_power_law(wave_opt_extended, A_blue, alpha_blue, A_red, alpha_red)

# Individual components (for plotting)
m_nev = gauss_lsf_flux(wave_opt_extended, LAM_NeV, popt_median_extended[0])
m_oii = gauss_lsf_flux(wave_opt_extended, LAM_OII, popt_median_extended[1])
m_neiii = gauss_lsf_flux(wave_opt_extended, LAM_NeIII, popt_median_extended[2])
m_hei = gauss_lsf_flux(wave_opt_extended, LAM_HeI, popt_median_extended[3])

m_heps = gauss_lsf_flux(wave_opt_extended, LAM_Hep, popt_median_extended[4])
m_hd = gauss_lsf_flux(wave_opt_extended, LAM_Hd, popt_median_extended[5])
m_hg = gauss_lsf_flux(wave_opt_extended, LAM_Hg, popt_median_extended[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_extended, LAM_OIII_aur, popt_median_extended[7])

m_hb = gauss_lsf_flux(wave_opt_extended, LAM_Hb, popt_median_extended[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_extended, LAM_OIII_1, popt_median_extended[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_extended, LAM_OIII_2, popt_median_extended[10])

m_ha = gauss_lsf_flux(wave_opt_extended, LAM_Ha, popt_median_extended[11])


# ----------------------------------------------------
# Plot: spectrum + residuals
# ----------------------------------------------------

fig, (ax, axr) = plt.subplots(2, 1,figsize=(10, 3),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_opt_extended,median_stacked_flux_opt_extended,where="mid",lw=1.2,color="#42A2B3",label="Median stack")

ax.fill_between(wave_opt_extended,median_stacked_flux_opt_extended - median_stacked_err_opt_extended,median_stacked_flux_opt_extended + median_stacked_err_opt_extended,alpha=0.3,color="#42A2B3")

ax.plot(wave_opt_extended, model_opt_median_extended, color="k", lw=1.2, label="Optical Model Fit")
ax.plot(wave_opt_extended, model_cont_median_extended, color="k", lw=1.2, linestyle = "dashed")

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.plot(wave_opt_extended, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_median_extended[0]/perr_median_extended[0]:.2f}")
ax.plot(wave_opt_extended, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_median_extended[1]/perr_median_extended[1]:.2f}")
ax.plot(wave_opt_extended, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_median_extended[2]/perr_median_extended[2]:.2f}")
ax.plot(wave_opt_extended, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_median_extended[3]/perr_median_extended[3]:.2f}")

ax.plot(wave_opt_extended, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_median_extended[4]/perr_median_extended[4]:.2f}")
ax.plot(wave_opt_extended, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_median_extended[5]/perr_median_extended[5]:.2f}")
ax.plot(wave_opt_extended, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_median_extended[6]/perr_median_extended[6]:.2f}")
ax.plot(wave_opt_extended, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_median_extended[7]/perr_median_extended[7]:.2f}")

ax.plot(wave_opt_extended, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_median_extended[8]/perr_median_extended[8]:.2f}")
ax.plot(wave_opt_extended, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_median_extended[9]/perr_median_extended[9]:.2f}")
ax.plot(wave_opt_extended, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_median_extended[10]/perr_median_extended[10]:.2f}")
ax.plot(wave_opt_extended, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_median_extended[11]/perr_median_extended[11]:.2f}")

ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Normalized Flux")

ax.legend(frameon=False,loc="center left",bbox_to_anchor=(1.02, 0.25),ncol=1)
ax.set_title("Optical extended Sources Median stack", pad=30)

# ---- bottom panel ----
resid = median_stacked_flux_opt_extended - model_opt_median_extended
axr.step(wave_opt_extended, resid / median_stacked_err_opt_extended, where="mid", lw=0.9, color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

plt.tight_layout(rect=[0, 0, 0.83, 1])
plt.show()

# ------  checking the significane of the line detections -------- :
# print(f"S/N  of Lya] = {popt_median_extended[1]/perr_median_extended[1]}")
# print(f"S/N  of NIV] = {popt_median_extended[2]/perr_median_extended[2]}")
# print(f"S/N  of CIV = {popt_median_extended[3]/perr_median_extended[3]}")
# print(f"S/N  of HeII = {popt_median_extended[4]/perr_median_extended[4]}")
# print(f"S/N  of OIII] = {popt_median_extended[5]/perr_median_extended[5]}")
# print(f"S/N  of NIII] = {popt_median_extended[6]/perr_median_extended[6]}")
# print(f"S/N  of CIII] = {popt_median_extended[7]/perr_median_extended[7]}")


## Mean Stack for Extended Sources

In [ ]:
z_med_extended = np.nanmedian(extended["z_Spec_1"])
wave_common_extended = make_prism_rest_wave_grid(lam_min_rest=0.1, lam_max_rest=0.8, z_med=z_med_extended, R_interp=R_interp)

opt_min, opt_max = 0.3, 0.7

mask_opt = (wave_common_extended > opt_min) & (wave_common_extended < opt_max)
wave_opt_extended = wave_common_extended[mask_opt]

f_extended = []
err_extended = []

for i,row in enumerate(extended[:]):
    print(i+1)
    z = row["z_Spec_1"]
    with fits.open(row["file_prism"]) as hdu:

        tab = Table(hdu[2].data) # using the EXTRACT3PIX1D extension because 3 pixels have better S/N
        wave = tab["WAVELENGTH"]/(1+z) # this is in microns
        flux = tab["FLUX"]*1e20 # erg s-1 cm-2 AA-1
        flux_err = tab["FLUX_ERR"]*1e20
    
    if row["O3_5007_combined_flux"]:
        norm = 1
    else:
        norm = 1

    norm = continuum_norm(wave, flux)
    flux_rs, flux_err_rs = spectres(wave_common_extended, wave, flux, spec_errs=flux_err, fill=np.nan)
    flux_norm = flux_rs/norm 
    flux_norm_err = flux_err_rs/norm
    
    wave = wave_common_extended
    flux = flux_norm 
    err = flux_norm_err 
    mask = ((wave > 0.3) & (wave < 0.7))# & np.isfinite(flux) & np.isfinite(err)) 
    
    lam = wave[mask] 
    f = flux[mask] 
    e = err[mask]
    print(row["ID"])
    
    # ----- sigma clipping ------- :
    clip_region = np.isfinite(f) & np.isfinite(e)
    clip_mask = np.zeros_like(f, dtype=bool)

    if np.any(clip_region):
        clipped = sigma_clip(
            f[clip_region],
            sigma=5.0,
            maxiters=5, 
            cenfunc="median",
            stdfunc=mad_std,
            masked=True
        ).mask
        clip_mask[clip_region] = np.asarray(clipped)

    f_clipped = f.copy()
    e_clipped = e.copy()
    f_clipped[clip_mask] = np.nan
    e_clipped[clip_mask] = np.nan

    # f = f_clipped - cont_model
    f = f_clipped
    e = e_clipped
    
    fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.5),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})
    
    # -------------------
    # Top: spectrum + fit
    # -------------------
    
    # observed spectrum
    ax.step(lam, f, where="mid", lw=1, color=blue, label="Spectrum")
    ax.fill_between(lam, f - e, f + e, color=blue, alpha=0.3)
    
    # continuum model
    # ax.plot(lam,cont_model,color="k",lw=1.2,alpha=0.9,label="Continuum fit")
    
    # points actually used in the fit
    # ax.scatter(lam[mask],f[mask],s=8,color="k",zorder=5,label="Fit region")
    
    # plotting the continuum subtracted spectra
    ax.step(lam,f,where="mid",lw=1,color="#B34280",label="Continuum-subtracted")
    ax.fill_between(lam,f - e,f + e,alpha=0.3, color = "#B34280")
    ax.axhline(0.0, color="gray", lw=0.8)

    label_emission_lines(ax, optical_emission_lines, y_frac=1.05)
    ax.set_ylabel(r"Normalized flux$_\lambda$")
    ax.legend(frameon=False, fontsize=8)
    
    # -------------------
    # Bottom: residuals (continuum subtracted)
    # -------------------
    
    axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
    axr.axhline(1.0, color="gray", ls="--", lw=0.6)
    axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
    
    axr.step(lam,f / e,where="mid",color="#9E3C55",lw=0.8)
    
    axr.set_ylabel(r"$(f - c)/\sigma$")
    axr.set_xlabel(r"Rest wavelength ($\mu$m)")
    axr.set_ylim(-8, 8)


    ax.set_title(f"Continuum Fits for extended Prism Source {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)

    
    plt.tight_layout()
    plt.show()

    # ------- saving the continuum subtarcted flux ------- :
    
    # Create opt-only arrays
    f_opt = np.full_like(wave_opt_extended, np.nan)
    e_opt = np.full_like(wave_opt_extended, np.nan)
    
    # Map lam → wave_opt_extended (they are identical grids if you used spectres earlier)
    f_opt[:] = f
    e_opt[:] = e
    
    f_extended.append(f_opt)
    err_extended.append(e_opt)
        
f_extended = np.array(f_extended)   # (Nspec, Nopt)
err_extended = np.array(err_extended)

# --- variance-weighted mean stack ---
# weights are inverse variance; avoid divide-by-zero
w = 1.0 / np.where(err_extended > 0, err_extended**2, np.nan)

# --- variance-weighted mean stack ---

w = 1.0 / np.where(err_extended > 0, err_extended**2, np.nan) # weights are inverse variance; avoid divide-by-zero
w_norm = w / np.nansum(w, axis=0) # weights should be normalized!

mean_stacked_flux_opt_extended = np.nansum(w_norm * f_extended, axis=0)
mean_stacked_err_opt_extended  = np.sqrt(1.0 / np.nansum(w, axis=0))

mean_stacked_flux_opt_extended = mean_stacked_flux_opt_extended
mean_stacked_err_opt_extended = mean_stacked_err_opt_extended

mean_stacked_resid_opt_extended = mean_stacked_flux_opt_extended / mean_stacked_err_opt_extended
mean_stacked_n_opt_extended = np.sum(np.isfinite(f_extended), axis=0)

fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.8),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# -------------------
# Top: opt stack
# -------------------

ax.step(wave_opt_extended,mean_stacked_flux_opt_extended,where="mid",lw=1.2,label="Mean stack", color = "#42A2B3")

ax.fill_between(wave_opt_extended,mean_stacked_flux_opt_extended - mean_stacked_err_opt_extended,mean_stacked_flux_opt_extended + mean_stacked_err_opt_extended,alpha=0.3, color = "#42A2B3")

# ax.axhline(0.0, color="gray", lw=0.8)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.set_ylabel(r"Mean flux")
ax.legend(frameon=False, fontsize=8)

# -------------------
# Bottom: residuals
# -------------------

axr.step(wave_opt_extended,mean_stacked_resid_opt_extended,where="mid",lw=0.9,color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")
axr.set_ylim(-8, 8)

ax.set_title("opt continuum-subtracted extended mean stack", pad =35)
plt.tight_layout()
plt.show()

# ------ fitting the mean stack ------ :

lam = wave_opt_extended
f   = mean_stacked_flux_opt_extended
e   = mean_stacked_err_opt_extended


# ------- line model ------- :

p0 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, np.nanmedian(f), -2.0, np.nanmedian(f), -1.0]
bounds = ([0,0, 0,0,0,0, 0,0,0, 0,0,0, 0, -np.inf, 0, -np.inf], 
          [np.inf, np.inf, np.inf, np.inf, np.inf,np.inf, np.inf, np.inf, np.inf, np.inf, np.inf,np.inf, np.inf, np.inf, np.inf, np.inf])

popt_mean_extended, pcov_mean_extended = curve_fit(
    optical_region_cont_fit_stack,
    lam, f,
    sigma=e,
    p0=p0,
    bounds=bounds,
    absolute_sigma=True
)

perr_mean_extended = np.sqrt(np.diag(pcov_mean_extended))

model_opt_mean_extended = optical_region_cont_fit_stack(wave_opt_extended, *popt_mean_extended)

# ------ continuum model ------ :

A_blue = popt_mean_extended[-4]
alpha_blue = popt_mean_extended[-3]

A_red = popt_mean_extended[-2]
alpha_red = popt_mean_extended[-1]

model_cont_mean_extended = opt_power_law(wave_opt_extended, A_blue, alpha_blue, A_red, alpha_red)

# Individual components (for plotting)
m_nev = gauss_lsf_flux(wave_opt_extended, LAM_NeV, popt_mean_extended[0])
m_oii = gauss_lsf_flux(wave_opt_extended, LAM_OII, popt_mean_extended[1])
m_neiii = gauss_lsf_flux(wave_opt_extended, LAM_NeIII, popt_mean_extended[2])
m_hei = gauss_lsf_flux(wave_opt_extended, LAM_HeI, popt_mean_extended[3])

m_heps = gauss_lsf_flux(wave_opt_extended, LAM_Hep, popt_mean_extended[4])
m_hd = gauss_lsf_flux(wave_opt_extended, LAM_Hd, popt_mean_extended[5])
m_hg = gauss_lsf_flux(wave_opt_extended, LAM_Hg, popt_mean_extended[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_extended, LAM_OIII_aur, popt_mean_extended[7])

m_hb = gauss_lsf_flux(wave_opt_extended, LAM_Hb, popt_mean_extended[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_extended, LAM_OIII_1, popt_mean_extended[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_extended, LAM_OIII_2, popt_mean_extended[10])

m_ha = gauss_lsf_flux(wave_opt_extended, LAM_Ha, popt_mean_extended[11])

# ----------------------------------------------------
# Plot: spectrum + residuals
# ----------------------------------------------------

fig, (ax, axr) = plt.subplots(2, 1,figsize=(10, 3),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_opt_extended,mean_stacked_flux_opt_extended,where="mid",lw=1.2,color="#42A2B3",label="mean stack")

ax.fill_between(wave_opt_extended,mean_stacked_flux_opt_extended - mean_stacked_err_opt_extended,mean_stacked_flux_opt_extended + mean_stacked_err_opt_extended,alpha=0.3,color="#42A2B3")

ax.plot(wave_opt_extended, model_opt_mean_extended, color="k", lw=1.2, label="Optical Model Fit")
ax.plot(wave_opt_extended, model_cont_mean_extended, color="k", lw=1.2, linestyle = "dashed")

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.plot(wave_opt_extended, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_mean_extended[0]/perr_mean_extended[0]:.2f}")
ax.plot(wave_opt_extended, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_mean_extended[1]/perr_mean_extended[1]:.2f}")
ax.plot(wave_opt_extended, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_mean_extended[2]/perr_mean_extended[2]:.2f}")
ax.plot(wave_opt_extended, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_mean_extended[3]/perr_mean_extended[3]:.2f}")

ax.plot(wave_opt_extended, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_mean_extended[4]/perr_mean_extended[4]:.2f}")
ax.plot(wave_opt_extended, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_mean_extended[5]/perr_mean_extended[5]:.2f}")
ax.plot(wave_opt_extended, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_mean_extended[6]/perr_mean_extended[6]:.2f}")
ax.plot(wave_opt_extended, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_mean_extended[7]/perr_mean_extended[7]:.2f}")

ax.plot(wave_opt_extended, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_mean_extended[8]/perr_mean_extended[8]:.2f}")
ax.plot(wave_opt_extended, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_mean_extended[9]/perr_mean_extended[9]:.2f}")
ax.plot(wave_opt_extended, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_mean_extended[10]/perr_mean_extended[10]:.2f}")
ax.plot(wave_opt_extended, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_mean_extended[11]/perr_mean_extended[11]:.2f}")


ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Normalized Flux")

ax.legend(frameon=False,loc="center left",bbox_to_anchor=(1.02, 0.25),ncol=1)
ax.set_title("Optical extended Sources Mean stack", pad=30)

# ---- bottom panel ----
resid = mean_stacked_flux_opt_extended - model_opt_mean_extended
axr.step(wave_opt_extended, resid / mean_stacked_err_opt_extended, where="mid", lw=0.9, color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

plt.tight_layout(rect=[0, 0, 0.83, 1])
plt.show()


# Intermediate Sources

In [ ]:
intermediate

## Median Stacks for Intermediate Sources

In [ ]:
z_med_intermediate = np.nanmedian(intermediate["z_Spec_1"])
wave_common_intermediate = make_prism_rest_wave_grid(lam_min_rest=0.1, lam_max_rest=0.8, z_med=z_med_intermediate, R_interp=R_interp)

opt_min, opt_max = 0.3, 0.7

mask_opt = (wave_common_intermediate > opt_min) & (wave_common_intermediate < opt_max)
wave_opt_intermediate = wave_common_intermediate[mask_opt]

f_intermediate = []
err_intermediate = []
norm_intermediate = []

for i,row in enumerate(intermediate[:]):
    print(i+1)
    z = row["z_Spec_1"]
    with fits.open(row["file_prism"]) as hdu:

        tab = Table(hdu[2].data) # using the EXTRACT3PIX1D extension because 3 pixels have better S/N
        wave = tab["WAVELENGTH"]/(1+z) # this is in microns
        flux = tab["FLUX"]*1e20 # erg s-1 cm-2 AA-1
        flux_err = tab["FLUX_ERR"]*1e20
    
    if row["O3_5007_combined_flux"]:
        norm = 1
    else:
        norm = 1

    norm = continuum_norm(wave, flux)   
    norm_intermediate.append(norm)
    flux_rs, flux_err_rs = spectres(wave_common_intermediate, wave, flux, spec_errs=flux_err, fill=np.nan)
    flux_norm = flux_rs/norm 
    flux_norm_err = flux_err_rs/norm
    
    wave = wave_common_intermediate
    flux = flux_norm 
    err = flux_norm_err 
    mask = ((wave > 0.3) & (wave < 0.7))# & np.isfinite(flux) & np.isfinite(err)) 
    
    lam = wave[mask] 
    f = flux[mask] 
    e = err[mask]
    print(row["ID"])
        
    fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.5),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})
    
    # -------------------
    # Top: spectrum + fit
    # -------------------
    
    # observed spectrum
    ax.step(lam, f, where="mid", lw=1, color=blue, label="Spectrum")
    ax.fill_between(lam, f - e, f + e, color=blue, alpha=0.3)
    
    # continuum model
    # ax.plot(lam,cont_model,color="k",lw=1.2,alpha=0.9,label="Continuum fit")
    
    # points actually used in the fit
    # ax.scatter(lam[mask],f[mask],s=8,color="k",zorder=5,label="Fit region")
    
    # plotting the continuum subtracted spectra
    ax.step(lam,f,where="mid",lw=1,color="#B34280",label="Continuum-subtracted")
    ax.fill_between(lam,f - e,f + e,alpha=0.3, color = "#B34280")
    ax.axhline(0.0, color="gray", lw=0.8)

    label_emission_lines(ax, optical_emission_lines, y_frac=1.05)
    ax.set_ylabel(r"Normalized flux$_\lambda$")
    ax.legend(frameon=False, fontsize=8)
    
    # -------------------
    # Bottom: residuals (continuum subtracted)
    # -------------------
    
    axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
    axr.axhline(1.0, color="gray", ls="--", lw=0.6)
    axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
    
    axr.step(lam,f / e,where="mid",color="#9E3C55",lw=0.8)
    
    axr.set_ylabel(r"$(f - c)/\sigma$")
    axr.set_xlabel(r"Rest wavelength ($\mu$m)")
    axr.set_ylim(-8, 8)


    ax.set_title(f"Continuum Fits for intermediate Prism Source {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)

    
    plt.tight_layout()
    plt.show()

    # ------- saving the continuum subtarcted flux ------- :
    
    # Create OPT-only arrays
    f_opt = np.full_like(wave_opt_intermediate, np.nan)
    e_opt = np.full_like(wave_opt_intermediate, np.nan)
    
    # Map lam → wave_opt_intermediate (they are identical grids if you used spectres earlier)
    f_opt[:] = f
    e_opt[:] = e
    
    f_intermediate.append(f_opt)
    err_intermediate.append(e_opt)

In [ ]:
# ----- median stacking ------:
# %matplotlib widget
opt_min, opt_max = 0.3, 0.7
mask_opt = (wave_common_intermediate > opt_min) & (wave_common_intermediate < opt_max)
wave_opt_intermediate = wave_common_intermediate[mask_opt]

f_intermediate = np.array(f_intermediate)   # (Nspec, Nopt)
err_intermediate = np.array(err_intermediate)

median_stacked_flux_opt_intermediate = np.nanmedian(f_intermediate, axis=0)

rng = np.random.default_rng()
n_mc = 1000

stacked_mc = np.empty((n_mc, len(wave_opt_intermediate)))

for k in range(n_mc):
    flux_pert = rng.normal(f_intermediate, err_intermediate)
    stacked_mc[k] = np.nanmedian(flux_pert, axis=0)

median_stacked_err_opt_intermediate = np.nanstd(stacked_mc, axis=0)
median_stacked_n_opt_intermediate = np.sum(~np.isnan(f_intermediate), axis=0)

median_stacked_flux_opt_intermediate = median_stacked_flux_opt_intermediate
median_stacked_err_opt_intermediate = median_stacked_err_opt_intermediate

median_stacked_resid_opt_intermediate = median_stacked_flux_opt_intermediate / median_stacked_err_opt_intermediate


fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.8),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# -------------------
# Top: opt stack
# -------------------

ax.step(wave_opt_intermediate,median_stacked_flux_opt_intermediate,where="mid",lw=1.2,label="Median stack (cont-sub)", color = "#42A2B3")

ax.fill_between(wave_opt_intermediate,median_stacked_flux_opt_intermediate - median_stacked_err_opt_intermediate,median_stacked_flux_opt_intermediate + median_stacked_err_opt_intermediate,alpha=0.3, color = "#42A2B3")

ax.axhline(0.0, color="gray", lw=0.8)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.set_ylabel(r"Normalized flux")
ax.legend(frameon=False, fontsize=8)

# -------------------
# Bottom: residuals
# -------------------

axr.step(wave_opt_intermediate,median_stacked_resid_opt_intermediate,where="mid",lw=0.9,color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")
axr.set_ylim(-8, 8)

# ax.set_title("opt continuum-subtracted intermediate median stack", pad =35)
ax.set_title("opt continuum intermediate median stack", pad =35)
plt.tight_layout()
plt.show()


# ------- fitting the median stack ------ :

lam = wave_opt_intermediate
f   = median_stacked_flux_opt_intermediate
e   = median_stacked_err_opt_intermediate

# ------- line model ------- :

p0 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, np.nanmedian(f), -2.0, np.nanmedian(f), -1.0]
bounds = ([0,0, 0,0,0,0, 0,0,0, 0,0,0, 0, -np.inf, 0, -np.inf], 
          [np.inf, np.inf, np.inf, np.inf, np.inf,np.inf, np.inf, np.inf, np.inf, np.inf, np.inf,np.inf, np.inf, np.inf, np.inf, np.inf])

print(z_med)

popt_median_intermediate, pcov_median_intermediate = curve_fit(
    optical_region_cont_fit_stack,
    lam, f,
    sigma=e,
    p0=p0,
    bounds=bounds,
    absolute_sigma=True
)

perr_median_intermediate = np.sqrt(np.diag(pcov_median_intermediate))

model_opt_median_intermediate = optical_region_cont_fit_stack(wave_opt_intermediate, *popt_median_intermediate)

# ------ continuum model ------ :

A_blue = popt_median_intermediate[-4]
alpha_blue = popt_median_intermediate[-3]

A_red = popt_median_intermediate[-2]
alpha_red = popt_median_intermediate[-1]

model_cont_median_intermediate = opt_power_law(wave_opt_intermediate, A_blue, alpha_blue, A_red, alpha_red)

# Individual components (for plotting)
m_nev = gauss_lsf_flux(wave_opt_intermediate, LAM_NeV, popt_median_intermediate[0])
m_oii = gauss_lsf_flux(wave_opt_intermediate, LAM_OII, popt_median_intermediate[1])
m_neiii = gauss_lsf_flux(wave_opt_intermediate, LAM_NeIII, popt_median_intermediate[2])
m_hei = gauss_lsf_flux(wave_opt_intermediate, LAM_HeI, popt_median_intermediate[3])

m_heps = gauss_lsf_flux(wave_opt_intermediate, LAM_Hep, popt_median_intermediate[4])
m_hd = gauss_lsf_flux(wave_opt_intermediate, LAM_Hd, popt_median_intermediate[5])
m_hg = gauss_lsf_flux(wave_opt_intermediate, LAM_Hg, popt_median_intermediate[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_aur, popt_median_intermediate[7])

m_hb = gauss_lsf_flux(wave_opt_intermediate, LAM_Hb, popt_median_intermediate[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_1, popt_median_intermediate[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_2, popt_median_intermediate[10])

m_ha = gauss_lsf_flux(wave_opt_intermediate, LAM_Ha, popt_median_intermediate[11])


# ----------------------------------------------------
# Plot: spectrum + residuals
# ----------------------------------------------------

fig, (ax, axr) = plt.subplots(2, 1,figsize=(10, 3),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_opt_intermediate,median_stacked_flux_opt_intermediate,where="mid",lw=1.2,color="#42A2B3",label="Median stack")

ax.fill_between(wave_opt_intermediate,median_stacked_flux_opt_intermediate - median_stacked_err_opt_intermediate,median_stacked_flux_opt_intermediate + median_stacked_err_opt_intermediate,alpha=0.3,color="#42A2B3")

ax.plot(wave_opt_intermediate, model_opt_median_intermediate, color="k", lw=1.2, label="Optical Model Fit")
ax.plot(wave_opt_intermediate, model_cont_median_intermediate, color="k", lw=1.2, linestyle = "dashed")

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.plot(wave_opt_intermediate, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_median_intermediate[0]/perr_median_intermediate[0]:.2f}")
ax.plot(wave_opt_intermediate, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_median_intermediate[1]/perr_median_intermediate[1]:.2f}")
ax.plot(wave_opt_intermediate, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_median_intermediate[2]/perr_median_intermediate[2]:.2f}")
ax.plot(wave_opt_intermediate, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_median_intermediate[3]/perr_median_intermediate[3]:.2f}")

ax.plot(wave_opt_intermediate, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_median_intermediate[4]/perr_median_intermediate[4]:.2f}")
ax.plot(wave_opt_intermediate, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_median_intermediate[5]/perr_median_intermediate[5]:.2f}")
ax.plot(wave_opt_intermediate, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_median_intermediate[6]/perr_median_intermediate[6]:.2f}")
ax.plot(wave_opt_intermediate, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_median_intermediate[7]/perr_median_intermediate[7]:.2f}")

ax.plot(wave_opt_intermediate, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_median_intermediate[8]/perr_median_intermediate[8]:.2f}")
ax.plot(wave_opt_intermediate, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_median_intermediate[9]/perr_median_intermediate[9]:.2f}")
ax.plot(wave_opt_intermediate, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_median_intermediate[10]/perr_median_intermediate[10]:.2f}")
ax.plot(wave_opt_intermediate, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_median_intermediate[11]/perr_median_intermediate[11]:.2f}")

ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Normalized Flux")

ax.legend(frameon=False,loc="center left",bbox_to_anchor=(1.02, 0.25),ncol=1)
ax.set_title("Optical intermediate Sources Median stack", pad=30)

# ---- bottom panel ----
resid = median_stacked_flux_opt_intermediate - model_opt_median_intermediate
axr.step(wave_opt_intermediate, resid / median_stacked_err_opt_intermediate, where="mid", lw=0.9, color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

plt.tight_layout(rect=[0, 0, 0.83, 1])
plt.show()

# ------  checking the significane of the line detections -------- :
# print(f"S/N  of Lya] = {popt_median_intermediate[1]/perr_median_intermediate[1]}")
# print(f"S/N  of NIV] = {popt_median_intermediate[2]/perr_median_intermediate[2]}")
# print(f"S/N  of CIV = {popt_median_intermediate[3]/perr_median_intermediate[3]}")
# print(f"S/N  of HeII = {popt_median_intermediate[4]/perr_median_intermediate[4]}")
# print(f"S/N  of OIII] = {popt_median_intermediate[5]/perr_median_intermediate[5]}")
# print(f"S/N  of NIII] = {popt_median_intermediate[6]/perr_median_intermediate[6]}")
# print(f"S/N  of CIII] = {popt_median_intermediate[7]/perr_median_intermediate[7]}")


## Mean Stacks for Intermediate Sources

In [ ]:
z_med_intermediate = np.nanmedian(intermediate["z_Spec_1"])
wave_common_intermediate = make_prism_rest_wave_grid(lam_min_rest=0.1, lam_max_rest=0.8, z_med=z_med_intermediate, R_interp=R_interp)

opt_min, opt_max = 0.3, 0.7

mask_opt = (wave_common_intermediate > opt_min) & (wave_common_intermediate < opt_max)
wave_opt_intermediate = wave_common_intermediate[mask_opt]

f_intermediate = []
err_intermediate = []

for i,row in enumerate(intermediate[:]):
    print(i+1)
    z = row["z_Spec_1"]
    with fits.open(row["file_prism"]) as hdu:

        tab = Table(hdu[2].data) # using the EXTRACT3PIX1D extension because 3 pixels have better S/N
        wave = tab["WAVELENGTH"]/(1+z) # this is in microns
        flux = tab["FLUX"]*1e20 # erg s-1 cm-2 AA-1
        flux_err = tab["FLUX_ERR"]*1e20
    
    if row["O3_5007_combined_flux"]:
        norm = 1
    else:
        norm = 1

    norm = continuum_norm(wave, flux)
    flux_rs, flux_err_rs = spectres(wave_common_intermediate, wave, flux, spec_errs=flux_err, fill=np.nan)
    flux_norm = flux_rs/norm 
    flux_norm_err = flux_err_rs/norm
    
    wave = wave_common_intermediate
    flux = flux_norm 
    err = flux_norm_err 
    mask = ((wave > 0.3) & (wave < 0.7))# & np.isfinite(flux) & np.isfinite(err)) 
    
    lam = wave[mask] 
    f = flux[mask] 
    e = err[mask]
    print(row["ID"])
    
    # ----- sigma clipping ------- :
    clip_region = np.isfinite(f) & np.isfinite(e)
    clip_mask = np.zeros_like(f, dtype=bool)

    if np.any(clip_region):
        clipped = sigma_clip(
            f[clip_region],
            sigma=5.0,
            maxiters=5, 
            cenfunc="median",
            stdfunc=mad_std,
            masked=True
        ).mask
        clip_mask[clip_region] = np.asarray(clipped)

    f_clipped = f.copy()
    e_clipped = e.copy()
    f_clipped[clip_mask] = np.nan
    e_clipped[clip_mask] = np.nan

    # f = f_clipped - cont_model
    f = f_clipped
    e = e_clipped
    
    fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.5),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})
    
    # -------------------
    # Top: spectrum + fit
    # -------------------
    
    # observed spectrum
    ax.step(lam, f, where="mid", lw=1, color=blue, label="Spectrum")
    ax.fill_between(lam, f - e, f + e, color=blue, alpha=0.3)
    
    # continuum model
    # ax.plot(lam,cont_model,color="k",lw=1.2,alpha=0.9,label="Continuum fit")
    
    # points actually used in the fit
    # ax.scatter(lam[mask],f[mask],s=8,color="k",zorder=5,label="Fit region")
    
    # plotting the continuum subtracted spectra
    ax.step(lam,f,where="mid",lw=1,color="#B34280",label="Continuum-subtracted")
    ax.fill_between(lam,f - e,f + e,alpha=0.3, color = "#B34280")
    ax.axhline(0.0, color="gray", lw=0.8)

    label_emission_lines(ax, optical_emission_lines, y_frac=1.05)
    ax.set_ylabel(r"Normalized flux$_\lambda$")
    ax.legend(frameon=False, fontsize=8)
    
    # -------------------
    # Bottom: residuals (continuum subtracted)
    # -------------------
    
    axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
    axr.axhline(1.0, color="gray", ls="--", lw=0.6)
    axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
    
    axr.step(lam,f / e,where="mid",color="#9E3C55",lw=0.8)
    
    axr.set_ylabel(r"$(f - c)/\sigma$")
    axr.set_xlabel(r"Rest wavelength ($\mu$m)")
    axr.set_ylim(-8, 8)


    ax.set_title(f"Continuum Fits for intermediate Prism Source {row['ID']}, "f"z = {row['z_Spec_1']:.3f}", pad = 40)

    
    plt.tight_layout()
    plt.show()

    # ------- saving the continuum subtarcted flux ------- :
    
    # Create opt-only arrays
    f_opt = np.full_like(wave_opt_intermediate, np.nan)
    e_opt = np.full_like(wave_opt_intermediate, np.nan)
    
    # Map lam → wave_opt_intermediate (they are identical grids if you used spectres earlier)
    f_opt[:] = f
    e_opt[:] = e
    
    f_intermediate.append(f_opt)
    err_intermediate.append(e_opt)
        
f_intermediate = np.array(f_intermediate)   # (Nspec, Nopt)
err_intermediate = np.array(err_intermediate)

# --- variance-weighted mean stack ---
# weights are inverse variance; avoid divide-by-zero
w = 1.0 / np.where(err_intermediate > 0, err_intermediate**2, np.nan)

# --- variance-weighted mean stack ---

w = 1.0 / np.where(err_intermediate > 0, err_intermediate**2, np.nan) # weights are inverse variance; avoid divide-by-zero
w_norm = w / np.nansum(w, axis=0) # weights should be normalized!

mean_stacked_flux_opt_intermediate = np.nansum(w_norm * f_intermediate, axis=0)
mean_stacked_err_opt_intermediate  = np.sqrt(1.0 / np.nansum(w, axis=0))

mean_stacked_flux_opt_intermediate = mean_stacked_flux_opt_intermediate
mean_stacked_err_opt_intermediate = mean_stacked_err_opt_intermediate

mean_stacked_resid_opt_intermediate = mean_stacked_flux_opt_intermediate / mean_stacked_err_opt_intermediate
mean_stacked_n_opt_intermediate = np.sum(np.isfinite(f_intermediate), axis=0)

fig, (ax, axr) = plt.subplots(2, 1,figsize=(5, 3.8),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# -------------------
# Top: opt stack
# -------------------

ax.step(wave_opt_intermediate,mean_stacked_flux_opt_intermediate,where="mid",lw=1.2,label="Mean stack", color = "#42A2B3")

ax.fill_between(wave_opt_intermediate,mean_stacked_flux_opt_intermediate - mean_stacked_err_opt_intermediate,mean_stacked_flux_opt_intermediate + mean_stacked_err_opt_intermediate,alpha=0.3, color = "#42A2B3")

# ax.axhline(0.0, color="gray", lw=0.8)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.set_ylabel(r"Mean flux")
ax.legend(frameon=False, fontsize=8)

# -------------------
# Bottom: residuals
# -------------------

axr.step(wave_opt_intermediate,mean_stacked_resid_opt_intermediate,where="mid",lw=0.9,color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")
axr.set_ylim(-8, 8)

ax.set_title("opt continuum-subtracted intermediate mean stack", pad =35)
plt.tight_layout()
plt.show()

# ------ fitting the mean stack ------ :

lam = wave_opt_intermediate
f   = mean_stacked_flux_opt_intermediate
e   = mean_stacked_err_opt_intermediate


# ------- line model ------- :

p0 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, np.nanmedian(f), -2.0, np.nanmedian(f), -1.0]
bounds = ([0,0, 0,0,0,0, 0,0,0, 0,0,0, 0, -np.inf, 0, -np.inf], 
          [np.inf, np.inf, np.inf, np.inf, np.inf,np.inf, np.inf, np.inf, np.inf, np.inf, np.inf,np.inf, np.inf, np.inf, np.inf, np.inf])

popt_mean_intermediate, pcov_mean_intermediate = curve_fit(
    optical_region_cont_fit_stack,
    lam, f,
    sigma=e,
    p0=p0,
    bounds=bounds,
    absolute_sigma=True
)

perr_mean_intermediate = np.sqrt(np.diag(pcov_mean_intermediate))

model_opt_mean_intermediate = optical_region_cont_fit_stack(wave_opt_intermediate, *popt_mean_intermediate)

# ------ continuum model ------ :

A_blue = popt_mean_intermediate[-4]
alpha_blue = popt_mean_intermediate[-3]

A_red = popt_mean_intermediate[-2]
alpha_red = popt_mean_intermediate[-1]

model_cont_mean_intermediate = opt_power_law(wave_opt_intermediate, A_blue, alpha_blue, A_red, alpha_red)

# Individual components (for plotting)
m_nev = gauss_lsf_flux(wave_opt_intermediate, LAM_NeV, popt_mean_intermediate[0])
m_oii = gauss_lsf_flux(wave_opt_intermediate, LAM_OII, popt_mean_intermediate[1])
m_neiii = gauss_lsf_flux(wave_opt_intermediate, LAM_NeIII, popt_mean_intermediate[2])
m_hei = gauss_lsf_flux(wave_opt_intermediate, LAM_HeI, popt_mean_intermediate[3])

m_heps = gauss_lsf_flux(wave_opt_intermediate, LAM_Hep, popt_mean_intermediate[4])
m_hd = gauss_lsf_flux(wave_opt_intermediate, LAM_Hd, popt_mean_intermediate[5])
m_hg = gauss_lsf_flux(wave_opt_intermediate, LAM_Hg, popt_mean_intermediate[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_aur, popt_mean_intermediate[7])

m_hb = gauss_lsf_flux(wave_opt_intermediate, LAM_Hb, popt_mean_intermediate[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_1, popt_mean_intermediate[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_2, popt_mean_intermediate[10])

m_ha = gauss_lsf_flux(wave_opt_intermediate, LAM_Ha, popt_mean_intermediate[11])

# ----------------------------------------------------
# Plot: spectrum + residuals
# ----------------------------------------------------

fig, (ax, axr) = plt.subplots(2, 1,figsize=(10, 3),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_opt_intermediate,mean_stacked_flux_opt_intermediate,where="mid",lw=1.2,color="#42A2B3",label="mean stack")

ax.fill_between(wave_opt_intermediate,mean_stacked_flux_opt_intermediate - mean_stacked_err_opt_intermediate,mean_stacked_flux_opt_intermediate + mean_stacked_err_opt_intermediate,alpha=0.3,color="#42A2B3")

ax.plot(wave_opt_intermediate, model_opt_mean_intermediate, color="k", lw=1.2, label="Optical Model Fit")
ax.plot(wave_opt_intermediate, model_cont_mean_intermediate, color="k", lw=1.2, linestyle = "dashed")

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

ax.plot(wave_opt_intermediate, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_mean_intermediate[0]/perr_mean_intermediate[0]:.2f}")
ax.plot(wave_opt_intermediate, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_mean_intermediate[1]/perr_mean_intermediate[1]:.2f}")
ax.plot(wave_opt_intermediate, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_mean_intermediate[2]/perr_mean_intermediate[2]:.2f}")
ax.plot(wave_opt_intermediate, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_mean_intermediate[3]/perr_mean_intermediate[3]:.2f}")

ax.plot(wave_opt_intermediate, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_mean_intermediate[4]/perr_mean_intermediate[4]:.2f}")
ax.plot(wave_opt_intermediate, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_mean_intermediate[5]/perr_mean_intermediate[5]:.2f}")
ax.plot(wave_opt_intermediate, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_mean_intermediate[6]/perr_mean_intermediate[6]:.2f}")
ax.plot(wave_opt_intermediate, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_mean_intermediate[7]/perr_mean_intermediate[7]:.2f}")

ax.plot(wave_opt_intermediate, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_mean_intermediate[8]/perr_mean_intermediate[8]:.2f}")
ax.plot(wave_opt_intermediate, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_mean_intermediate[9]/perr_mean_intermediate[9]:.2f}")
ax.plot(wave_opt_intermediate, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_mean_intermediate[10]/perr_mean_intermediate[10]:.2f}")
ax.plot(wave_opt_intermediate, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_mean_intermediate[11]/perr_mean_intermediate[11]:.2f}")


ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Normalized Flux")

ax.legend(frameon=False,loc="center left",bbox_to_anchor=(1.02, 0.25),ncol=1)
ax.set_title("Optical intermediate Sources Mean stack", pad=30)

# ---- bottom panel ----
resid = mean_stacked_flux_opt_intermediate - model_opt_mean_intermediate
axr.step(wave_opt_intermediate, resid / mean_stacked_err_opt_intermediate, where="mid", lw=0.9, color="#9E3C55")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

plt.tight_layout(rect=[0, 0, 0.83, 1])
plt.show()


# Compact and Extended Stacks

## Median Stacks

In [ ]:
# --------- compact median (urvi + sophia) ------- :

fig, (ax, axr) = plt.subplots(2, 1,figsize=(10, 4),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_uv_compact,median_stacked_flux_uv_compact,where="mid",lw=1,color="#31ACC4",label="Compact Median stack")
ax.fill_between(wave_uv_compact,median_stacked_flux_uv_compact - median_stacked_err_uv_compact,median_stacked_flux_uv_compact + median_stacked_err_uv_compact,alpha=0.1,color="#31ACC4")

# ax2 = ax.twinx()

ax.plot(wave_uv_compact, model_uv_median_compact, color="k", lw=1)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

# urvi line detections:
m_niv = gauss_lsf_flux(lam, LAM_NIV, popt_median_compact[1], popt_median_compact[0])
m_civ = gauss_lsf_flux(lam, LAM_CIV, popt_median_compact[2], popt_median_compact[0])
m_heii = gauss_lsf_flux(lam, LAM_HeII, popt_median_compact[3], popt_median_compact[0])
m_oiii = gauss_lsf_flux(lam, LAM_OIII, popt_median_compact[4], popt_median_compact[0])
m_niii = gauss_lsf_flux(lam, LAM_NIII, popt_median_compact[5], popt_median_compact[0])
m_ciii = gauss_lsf_flux(lam, LAM_CIII, popt_median_compact[6], popt_median_compact[0])

ax.plot(lam, m_niv, ls="--", lw=0.9, label=f"NIV], $\sigma$ = {popt_median_compact[1]/perr_median_compact[1]:.2f}", color="#31ACC4")
ax.plot(lam, m_civ, ls="--", lw=0.9, label=f"CIV, $\sigma$ = {popt_median_compact[2]/perr_median_compact[2]:.2f}", color="#31ACC4")
ax.plot(lam, m_heii, ls="--", lw=0.9, label=f"He II, $\sigma$ = {popt_median_compact[3]/perr_median_compact[3]:.2f}", color="#31ACC4")
ax.plot(lam, m_oiii, ls="--", lw=0.9, label=f"OIII], $\sigma$ = {popt_median_compact[4]/perr_median_compact[4]:.2f}", color="#31ACC4")
ax.plot(lam, m_niii, ls="--", lw=0.9, label=f"NIII], $\sigma$ = {popt_median_compact[5]/perr_median_compact[5]:.2f}", color="#31ACC4")
ax.plot(lam, m_ciii, ls="--", lw=0.9, label=f"CIII], $\sigma$ = {popt_median_compact[6]/perr_median_compact[6]:.2f}", color="#31ACC4")

ax.step(wave_uv_extended,median_stacked_flux_uv_extended,where="mid",lw=1,color="firebrick",label="Extended Median stack")
ax.fill_between(wave_uv_extended,median_stacked_flux_uv_extended - median_stacked_err_uv_extended,median_stacked_flux_uv_extended + median_stacked_err_uv_extended,alpha=0.1,color="firebrick")
ax.plot(wave_uv_extended, model_uv_median_extended, color="k", lw=1)


m_niv = gauss_lsf_flux(lam, LAM_NIV, popt_median_extended[1], popt_median_extended[0])
m_civ = gauss_lsf_flux(lam, LAM_CIV, popt_median_extended[2], popt_median_extended[0])
m_heii = gauss_lsf_flux(lam, LAM_HeII, popt_median_extended[3], popt_median_extended[0])
m_oiii = gauss_lsf_flux(lam, LAM_OIII, popt_median_extended[4], popt_median_extended[0])
m_niii = gauss_lsf_flux(lam, LAM_NIII, popt_median_extended[5], popt_median_extended[0])
m_ciii = gauss_lsf_flux(lam, LAM_CIII, popt_median_extended[6], popt_median_extended[0])

ax.plot(lam, m_niv, ls="--", lw=0.9, label=f"NIV], $\sigma$ = {popt_median_extended[1]/perr_median_extended[1]:.2f}", color="firebrick")
ax.plot(lam, m_civ, ls="--", lw=0.9, label=f"CIV, $\sigma$ = {popt_median_extended[2]/perr_median_extended[2]:.2f}", color="firebrick")
ax.plot(lam, m_heii, ls="--", lw=0.9, label=f"He II, $\sigma$ = {popt_median_extended[3]/perr_median_extended[3]:.2f}", color="firebrick")
ax.plot(lam, m_oiii, ls="--", lw=0.9, label=f"OIII], $\sigma$ = {popt_median_extended[4]/perr_median_extended[4]:.2f}", color="firebrick")
ax.plot(lam, m_niii, ls="--", lw=0.9, label=f"NIII], $\sigma$ = {popt_median_extended[5]/perr_median_extended[5]:.2f}", color="firebrick")
ax.plot(lam, m_ciii, ls="--", lw=0.9, label=f"CIII], $\sigma$ = {popt_median_extended[6]/perr_median_extended[6]:.2f}", color="firebrick")


ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Flux (arbitrary units)")
lines1, labels1 = ax.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon = False, fontsize =10, ncols =2)
ax.legend(lines1, labels1, loc="upper right", frameon = False, fontsize =10, ncols =2)
#
# ax.grid()
# ax.set_title("UV continuum-subtracted median stacks", pad=30)
ax.set_title("UV Compact/Extended Sources Median stacks", pad=30)
# ax.set_ylim(-0.5,2)
# ax2.set_ylim(-5, 35)


# ---- bottom panel ----
resid = median_stacked_flux_uv_compact - model_uv_median_compact
axr.step(wave_uv_compact, resid / median_stacked_err_uv_compact, where="mid", lw=0.9, color="#31ACC4")

resid = median_stacked_flux_uv_extended - model_uv_median_extended
axr.step(wave_uv_extended, resid / median_stacked_err_uv_extended, where="mid", lw=0.9, color="firebrick")


axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
axr.grid(alpha= 0.3)
axr.set_ylim(-8, 8)

plt.xlim(0.135,0.34)
    
axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

# ---- difference -----:
# axd.step(wave_uv_compact, median_stacked_flux_uv_compact , where = "mid", lw = 1, color="gray",label="Median Stack - difference") 
# axd.set_ylabel(r"Flux")
# axd.set_xlabel(r"Rest wavelength ($\mu$m)")
# axd.axhline(0.0, color="gray", lw=0.8)
# axd.legend()
# axd.grid()

plt.tight_layout()
plt.show()

In [ ]:
# --------- mean (urvi + sophia) ------- :

fig, (ax, axr) = plt.subplots(2, 1,figsize=(10, 4),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_uv_compact,mean_stacked_flux_uv_compact,where="mid",lw=1,color="#31ACC4",label="Compact mean stack")
ax.fill_between(wave_uv_compact,mean_stacked_flux_uv_compact - mean_stacked_err_uv_compact,mean_stacked_flux_uv_compact + mean_stacked_err_uv_compact,alpha=0.1,color="#31ACC4")

# ax2 = ax.twinx()

ax.plot(wave_uv_compact, model_uv_mean_compact, color="k", lw=1)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

# urvi line detections:
m_niv = gauss_lsf_flux(lam, LAM_NIV, popt_mean_compact[1], popt_mean_compact[0])
m_civ = gauss_lsf_flux(lam, LAM_CIV, popt_mean_compact[2], popt_mean_compact[0])
m_heii = gauss_lsf_flux(lam, LAM_HeII, popt_mean_compact[3], popt_mean_compact[0])
m_oiii = gauss_lsf_flux(lam, LAM_OIII, popt_mean_compact[4], popt_mean_compact[0])
m_niii = gauss_lsf_flux(lam, LAM_NIII, popt_mean_compact[5], popt_mean_compact[0])
m_ciii = gauss_lsf_flux(lam, LAM_CIII, popt_mean_compact[6], popt_mean_compact[0])

ax.plot(lam, m_niv, ls="--", lw=0.9, label=f"NIV], $\sigma$ = {popt_mean_compact[1]/perr_mean_compact[1]:.2f}", color="#31ACC4")
ax.plot(lam, m_civ, ls="--", lw=0.9, label=f"CIV, $\sigma$ = {popt_mean_compact[2]/perr_mean_compact[2]:.2f}", color="#31ACC4")
ax.plot(lam, m_heii, ls="--", lw=0.9, label=f"He II, $\sigma$ = {popt_mean_compact[3]/perr_mean_compact[3]:.2f}", color="#31ACC4")
ax.plot(lam, m_oiii, ls="--", lw=0.9, label=f"OIII], $\sigma$ = {popt_mean_compact[4]/perr_mean_compact[4]:.2f}", color="#31ACC4")
ax.plot(lam, m_niii, ls="--", lw=0.9, label=f"NIII], $\sigma$ = {popt_mean_compact[5]/perr_mean_compact[5]:.2f}", color="#31ACC4")
ax.plot(lam, m_ciii, ls="--", lw=0.9, label=f"CIII], $\sigma$ = {popt_mean_compact[6]/perr_mean_compact[6]:.2f}", color="#31ACC4")

ax.step(wave_uv_extended,mean_stacked_flux_uv_extended,where="mid",lw=1,color="firebrick",label="Extended mean stack")
ax.fill_between(wave_uv_extended,mean_stacked_flux_uv_extended - mean_stacked_err_uv_extended,mean_stacked_flux_uv_extended + mean_stacked_err_uv_extended,alpha=0.1,color="firebrick")

ax.plot(wave_uv_extended, model_uv_mean_extended, color="k", lw=1)

m_niv = gauss_lsf_flux(lam, LAM_NIV, popt_mean_extended[1], popt_mean_extended[0])
m_civ = gauss_lsf_flux(lam, LAM_CIV, popt_mean_extended[2], popt_mean_extended[0])
m_heii = gauss_lsf_flux(lam, LAM_HeII, popt_mean_extended[3], popt_mean_extended[0])
m_oiii = gauss_lsf_flux(lam, LAM_OIII, popt_mean_extended[4], popt_mean_extended[0])
m_niii = gauss_lsf_flux(lam, LAM_NIII, popt_mean_extended[5], popt_mean_extended[0])
m_ciii = gauss_lsf_flux(lam, LAM_CIII, popt_mean_extended[6], popt_mean_extended[0])

ax.plot(lam, m_niv, ls="--", lw=0.9, label=f"NIV], $\sigma$ = {popt_mean_extended[1]/perr_mean_extended[1]:.2f}", color="firebrick")
ax.plot(lam, m_civ, ls="--", lw=0.9, label=f"CIV, $\sigma$ = {popt_mean_extended[2]/perr_mean_extended[2]:.2f}", color="firebrick")
ax.plot(lam, m_heii, ls="--", lw=0.9, label=f"He II, $\sigma$ = {popt_mean_extended[3]/perr_mean_extended[3]:.2f}", color="firebrick")
ax.plot(lam, m_oiii, ls="--", lw=0.9, label=f"OIII], $\sigma$ = {popt_mean_extended[4]/perr_mean_extended[4]:.2f}", color="firebrick")
ax.plot(lam, m_niii, ls="--", lw=0.9, label=f"NIII], $\sigma$ = {popt_mean_extended[5]/perr_mean_extended[5]:.2f}", color="firebrick")
ax.plot(lam, m_ciii, ls="--", lw=0.9, label=f"CIII], $\sigma$ = {popt_mean_extended[6]/perr_mean_extended[6]:.2f}", color="firebrick")


ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Flux (arbitrary units)")
lines1, labels1 = ax.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon = False, fontsize =10, ncols =2)
ax.legend(lines1, labels1, loc="upper right", frameon = False, fontsize =10, ncols =2)
# ax.set_title("UV continuum-subtracted mean stacks", pad=30)
ax.set_title("UV Compact/Extended Sources Mean stacks", pad=30)
ax.grid(alpha=0.1)


# ---- bottom panel ----
resid = mean_stacked_flux_uv_compact - model_uv_mean_compact
axr.step(wave_uv_compact, resid / mean_stacked_err_uv_compact, where="mid", lw=0.9, color="#31ACC4")

resid = mean_stacked_flux_uv_extended - model_uv_mean_extended
axr.step(wave_uv_extended, resid / mean_stacked_err_uv_extended, where="mid", lw=0.9, color="firebrick")


axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)

axr.set_ylim(-8, 8)
plt.xlim(0.135,0.34)
    
axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")
axr.grid(alpha=0.3)

# ---- difference -----:
# axd.step(wave_uv_compact, mean_stacked_flux_uv_compact , where = "mid", lw = 1, color="gray",label="Mean Stack - difference") 
# axd.set_ylabel(r"Flux")
# axd.set_xlabel(r"Rest wavelength ($\mu$m)")
# axd.axhline(0.0, color="gray", lw=0.8)
# axd.legend()
# axd.grid()

plt.tight_layout()
plt.show()

# Compact, Extended and Intermediate Stacks

## Median Stacks

In [ ]:
# --------- compact median (urvi + sophia) ------- :

fig, (ax, axr) = plt.subplots(2, 1,figsize=(15, 4),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_opt_compact,median_stacked_flux_opt_compact,where="mid",lw=1,color="#31ACC4",label="Compact Median stack")
ax.fill_between(wave_opt_compact,median_stacked_flux_opt_compact - median_stacked_err_opt_compact,median_stacked_flux_opt_compact + median_stacked_err_opt_compact,alpha=0.1,color="#31ACC4")

# ax2 = ax.twinx()

ax.plot(wave_opt_compact, model_opt_median_compact, color="k", lw=1)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

# compact sources:

m_nev = gauss_lsf_flux(wave_opt_compact, LAM_NeV, popt_median_compact[0])
m_oii = gauss_lsf_flux(wave_opt_compact, LAM_OII, popt_median_compact[1])
m_neiii = gauss_lsf_flux(wave_opt_compact, LAM_NeIII, popt_median_compact[2])
m_hei = gauss_lsf_flux(wave_opt_compact, LAM_HeI, popt_median_compact[3])

m_heps = gauss_lsf_flux(wave_opt_compact, LAM_Hep, popt_median_compact[4])
m_hd = gauss_lsf_flux(wave_opt_compact, LAM_Hd, popt_median_compact[5])
m_hg = gauss_lsf_flux(wave_opt_compact, LAM_Hg, popt_median_compact[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_compact, LAM_OIII_aur, popt_median_compact[7])

m_hb = gauss_lsf_flux(wave_opt_compact, LAM_Hb, popt_median_compact[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_compact, LAM_OIII_1, popt_median_compact[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_compact, LAM_OIII_2, popt_median_compact[10])

m_ha = gauss_lsf_flux(wave_opt_compact, LAM_Ha, popt_median_compact[11])

ax.plot(wave_opt_compact, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_median_compact[0]/perr_median_compact[0]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_median_compact[1]/perr_median_compact[1]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_median_compact[2]/perr_median_compact[2]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_median_compact[3]/perr_median_compact[3]:.2f}", color="#31ACC4")

ax.plot(wave_opt_compact, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_median_compact[4]/perr_median_compact[4]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_median_compact[5]/perr_median_compact[5]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_median_compact[6]/perr_median_compact[6]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_median_compact[7]/perr_median_compact[7]:.2f}", color="#31ACC4")

ax.plot(wave_opt_compact, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_median_compact[8]/perr_median_compact[8]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_median_compact[9]/perr_median_compact[9]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_median_compact[10]/perr_median_compact[10]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_median_compact[11]/perr_median_compact[11]:.2f}", color="#31ACC4")

# extended sources :

ax.step(wave_opt_extended,median_stacked_flux_opt_extended,where="mid",lw=1,color="firebrick",label="Extended Median stack")
ax.fill_between(wave_opt_extended,median_stacked_flux_opt_extended - median_stacked_err_opt_extended,median_stacked_flux_opt_extended + median_stacked_err_opt_extended,alpha=0.1,color="firebrick")
ax.plot(wave_opt_extended, model_opt_median_extended, color="k", lw=1)

m_nev = gauss_lsf_flux(wave_opt_extended, LAM_NeV, popt_median_extended[0])
m_oii = gauss_lsf_flux(wave_opt_extended, LAM_OII, popt_median_extended[1])
m_neiii = gauss_lsf_flux(wave_opt_extended, LAM_NeIII, popt_median_extended[2])
m_hei = gauss_lsf_flux(wave_opt_extended, LAM_HeI, popt_median_extended[3])

m_heps = gauss_lsf_flux(wave_opt_extended, LAM_Hep, popt_median_extended[4])
m_hd = gauss_lsf_flux(wave_opt_extended, LAM_Hd, popt_median_extended[5])
m_hg = gauss_lsf_flux(wave_opt_extended, LAM_Hg, popt_median_extended[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_extended, LAM_OIII_aur, popt_median_extended[7])

m_hb = gauss_lsf_flux(wave_opt_extended, LAM_Hb, popt_median_extended[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_extended, LAM_OIII_1, popt_median_extended[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_extended, LAM_OIII_2, popt_median_extended[10])

m_ha = gauss_lsf_flux(wave_opt_extended, LAM_Ha, popt_median_extended[11])

ax.plot(wave_opt_extended, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_median_extended[0]/perr_median_extended[0]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_median_extended[1]/perr_median_extended[1]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_median_extended[2]/perr_median_extended[2]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_median_extended[3]/perr_median_extended[3]:.2f}", color="firebrick")

ax.plot(wave_opt_extended, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_median_extended[4]/perr_median_extended[4]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_median_extended[5]/perr_median_extended[5]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_median_extended[6]/perr_median_extended[6]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_median_extended[7]/perr_median_extended[7]:.2f}", color="firebrick")

ax.plot(wave_opt_extended, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_median_extended[8]/perr_median_extended[8]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_median_extended[9]/perr_median_extended[9]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_median_extended[10]/perr_median_extended[10]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_median_extended[11]/perr_median_extended[11]:.2f}", color="firebrick")

# intermediate sources:

ax.step(wave_opt_intermediate,median_stacked_flux_opt_intermediate,where="mid",lw=1,color="#F5C242",label="Intermediate Median stack")
ax.fill_between(wave_opt_intermediate,median_stacked_flux_opt_intermediate - median_stacked_err_opt_intermediate,median_stacked_flux_opt_intermediate + median_stacked_err_opt_intermediate,alpha=0.1,color="#F5C242")
ax.plot(wave_opt_intermediate, model_opt_median_intermediate, color="k", lw=1)

m_nev = gauss_lsf_flux(wave_opt_intermediate, LAM_NeV, popt_median_intermediate[0])
m_oii = gauss_lsf_flux(wave_opt_intermediate, LAM_OII, popt_median_intermediate[1])
m_neiii = gauss_lsf_flux(wave_opt_intermediate, LAM_NeIII, popt_median_intermediate[2])
m_hei = gauss_lsf_flux(wave_opt_intermediate, LAM_HeI, popt_median_intermediate[3])

m_heps = gauss_lsf_flux(wave_opt_intermediate, LAM_Hep, popt_median_intermediate[4])
m_hd = gauss_lsf_flux(wave_opt_intermediate, LAM_Hd, popt_median_intermediate[5])
m_hg = gauss_lsf_flux(wave_opt_intermediate, LAM_Hg, popt_median_intermediate[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_aur, popt_median_intermediate[7])

m_hb = gauss_lsf_flux(wave_opt_intermediate, LAM_Hb, popt_median_intermediate[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_1, popt_median_intermediate[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_2, popt_median_intermediate[10])

m_ha = gauss_lsf_flux(wave_opt_intermediate, LAM_Ha, popt_median_intermediate[11])

ax.plot(wave_opt_intermediate, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_median_intermediate[0]/perr_median_intermediate[0]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_median_intermediate[1]/perr_median_intermediate[1]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_median_intermediate[2]/perr_median_intermediate[2]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_median_intermediate[3]/perr_median_intermediate[3]:.2f}", color = "#F5C242")

ax.plot(wave_opt_intermediate, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_median_intermediate[4]/perr_median_intermediate[4]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_median_intermediate[5]/perr_median_intermediate[5]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_median_intermediate[6]/perr_median_intermediate[6]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_median_intermediate[7]/perr_median_intermediate[7]:.2f}", color = "#F5C242")

ax.plot(wave_opt_intermediate, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_median_intermediate[8]/perr_median_intermediate[8]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_median_intermediate[9]/perr_median_intermediate[9]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_median_intermediate[10]/perr_median_intermediate[10]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_median_intermediate[11]/perr_median_intermediate[11]:.2f}", color = "#F5C242")

ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Normalized Flux (arbitrary units)")
lines1, labels1 = ax.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon = False, fontsize =10, ncols =2)
ax.legend(lines1, labels1, loc="upper right", frameon = False, fontsize =8, ncols =3)
#
# ax.grid()
# ax.set_title("opt continuum-subtracted median stacks", pad=30)
ax.set_title(f"Optical Compact/Extended/Intermediate Sources Median stacks {z_bins[0]}$<z<${z_bins[-1]}", pad=30)
# ax.set_ylim(-0.5,2)
# ax2.set_ylim(-5, 35)


# ---- bottom panel ----
resid = median_stacked_flux_opt_compact - model_opt_median_compact
axr.step(wave_opt_compact, resid / median_stacked_err_opt_compact, where="mid", lw=0.9, color="#31ACC4")

resid = median_stacked_flux_opt_extended - model_opt_median_extended
axr.step(wave_opt_extended, resid / median_stacked_err_opt_extended, where="mid", lw=0.9, color="firebrick")

resid = median_stacked_flux_opt_intermediate - model_opt_median_intermediate
axr.step(wave_opt_intermediate, resid / median_stacked_err_opt_intermediate, where="mid", lw=0.9, color="#F5C242")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
axr.grid(alpha= 0.3)
# axr.set_ylim(-8, 8)

plt.xlim(0.3,1)
    
axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

# ---- difference -----:
# axd.step(wave_opt_compact, median_stacked_flux_opt_compact , where = "mid", lw = 1, color="gray",label="Median Stack - difference") 
# axd.set_ylabel(r"Flux")
# axd.set_xlabel(r"Rest wavelength ($\mu$m)")
# axd.axhline(0.0, color="gray", lw=0.8)
# axd.legend()
# axd.grid()

plt.tight_layout()
plt.show()

## Mean Stacks

In [ ]:
# --------- mean stacks ------- :

fig, (ax, axr) = plt.subplots(2, 1,figsize=(15, 4),dpi=200,sharex=True,gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# ---- top panel ----
ax.step(wave_opt_compact,mean_stacked_flux_opt_compact,where="mid",lw=1,color="#31ACC4",label="Compact Mean stack")
ax.fill_between(wave_opt_compact,mean_stacked_flux_opt_compact - mean_stacked_err_opt_compact,mean_stacked_flux_opt_compact + mean_stacked_err_opt_compact,alpha=0.1,color="#31ACC4")

# ax2 = ax.twinx()

ax.plot(wave_opt_compact, model_opt_mean_compact, color="k", lw=1)

label_emission_lines(ax, optical_emission_lines, y_frac=1.05)

# compact sources:

m_nev = gauss_lsf_flux(wave_opt_compact, LAM_NeV, popt_mean_compact[0])
m_oii = gauss_lsf_flux(wave_opt_compact, LAM_OII, popt_mean_compact[1])
m_neiii = gauss_lsf_flux(wave_opt_compact, LAM_NeIII, popt_mean_compact[2])
m_hei = gauss_lsf_flux(wave_opt_compact, LAM_HeI, popt_mean_compact[3])

m_heps = gauss_lsf_flux(wave_opt_compact, LAM_Hep, popt_mean_compact[4])
m_hd = gauss_lsf_flux(wave_opt_compact, LAM_Hd, popt_mean_compact[5])
m_hg = gauss_lsf_flux(wave_opt_compact, LAM_Hg, popt_mean_compact[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_compact, LAM_OIII_aur, popt_mean_compact[7])

m_hb = gauss_lsf_flux(wave_opt_compact, LAM_Hb, popt_mean_compact[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_compact, LAM_OIII_1, popt_mean_compact[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_compact, LAM_OIII_2, popt_mean_compact[10])

m_ha = gauss_lsf_flux(wave_opt_compact, LAM_Ha, popt_mean_compact[11])

ax.plot(wave_opt_compact, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_mean_compact[0]/perr_mean_compact[0]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_mean_compact[1]/perr_mean_compact[1]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_mean_compact[2]/perr_mean_compact[2]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_mean_compact[3]/perr_mean_compact[3]:.2f}", color="#31ACC4")

ax.plot(wave_opt_compact, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_mean_compact[4]/perr_mean_compact[4]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_mean_compact[5]/perr_mean_compact[5]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_mean_compact[6]/perr_mean_compact[6]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_mean_compact[7]/perr_mean_compact[7]:.2f}", color="#31ACC4")

ax.plot(wave_opt_compact, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_mean_compact[8]/perr_mean_compact[8]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_mean_compact[9]/perr_mean_compact[9]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_mean_compact[10]/perr_mean_compact[10]:.2f}", color="#31ACC4")
ax.plot(wave_opt_compact, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_mean_compact[11]/perr_mean_compact[11]:.2f}", color="#31ACC4")

# extended sources :

ax.step(wave_opt_extended,mean_stacked_flux_opt_extended,where="mid",lw=1,color="firebrick",label="Extended Mean stack")
ax.fill_between(wave_opt_extended,mean_stacked_flux_opt_extended - mean_stacked_err_opt_extended,mean_stacked_flux_opt_extended + mean_stacked_err_opt_extended,alpha=0.1,color="firebrick")
ax.plot(wave_opt_extended, model_opt_mean_extended, color="k", lw=1)

m_nev = gauss_lsf_flux(wave_opt_extended, LAM_NeV, popt_mean_extended[0])
m_oii = gauss_lsf_flux(wave_opt_extended, LAM_OII, popt_mean_extended[1])
m_neiii = gauss_lsf_flux(wave_opt_extended, LAM_NeIII, popt_mean_extended[2])
m_hei = gauss_lsf_flux(wave_opt_extended, LAM_HeI, popt_mean_extended[3])

m_heps = gauss_lsf_flux(wave_opt_extended, LAM_Hep, popt_mean_extended[4])
m_hd = gauss_lsf_flux(wave_opt_extended, LAM_Hd, popt_mean_extended[5])
m_hg = gauss_lsf_flux(wave_opt_extended, LAM_Hg, popt_mean_extended[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_extended, LAM_OIII_aur, popt_mean_extended[7])

m_hb = gauss_lsf_flux(wave_opt_extended, LAM_Hb, popt_mean_extended[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_extended, LAM_OIII_1, popt_mean_extended[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_extended, LAM_OIII_2, popt_mean_extended[10])

m_ha = gauss_lsf_flux(wave_opt_extended, LAM_Ha, popt_mean_extended[11])

ax.plot(wave_opt_extended, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_mean_extended[0]/perr_mean_extended[0]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_mean_extended[1]/perr_mean_extended[1]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_mean_extended[2]/perr_mean_extended[2]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_mean_extended[3]/perr_mean_extended[3]:.2f}", color="firebrick")

ax.plot(wave_opt_extended, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_mean_extended[4]/perr_mean_extended[4]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_mean_extended[5]/perr_mean_extended[5]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_mean_extended[6]/perr_mean_extended[6]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_mean_extended[7]/perr_mean_extended[7]:.2f}", color="firebrick")

ax.plot(wave_opt_extended, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_mean_extended[8]/perr_mean_extended[8]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_mean_extended[9]/perr_mean_extended[9]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_mean_extended[10]/perr_mean_extended[10]:.2f}", color="firebrick")
ax.plot(wave_opt_extended, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_mean_extended[11]/perr_mean_extended[11]:.2f}", color="firebrick")

# intermediate sources:

ax.step(wave_opt_intermediate,mean_stacked_flux_opt_intermediate,where="mid",lw=1,color="#F5C242",label="Intermediate Mean stack")
ax.fill_between(wave_opt_intermediate,mean_stacked_flux_opt_intermediate - mean_stacked_err_opt_intermediate,mean_stacked_flux_opt_intermediate + mean_stacked_err_opt_intermediate,alpha=0.1,color="#F5C242")
ax.plot(wave_opt_intermediate, model_opt_mean_intermediate, color="k", lw=1)

m_nev = gauss_lsf_flux(wave_opt_intermediate, LAM_NeV, popt_mean_intermediate[0])
m_oii = gauss_lsf_flux(wave_opt_intermediate, LAM_OII, popt_mean_intermediate[1])
m_neiii = gauss_lsf_flux(wave_opt_intermediate, LAM_NeIII, popt_mean_intermediate[2])
m_hei = gauss_lsf_flux(wave_opt_intermediate, LAM_HeI, popt_mean_intermediate[3])

m_heps = gauss_lsf_flux(wave_opt_intermediate, LAM_Hep, popt_mean_intermediate[4])
m_hd = gauss_lsf_flux(wave_opt_intermediate, LAM_Hd, popt_mean_intermediate[5])
m_hg = gauss_lsf_flux(wave_opt_intermediate, LAM_Hg, popt_mean_intermediate[6])
m_oiii_aur = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_aur, popt_mean_intermediate[7])

m_hb = gauss_lsf_flux(wave_opt_intermediate, LAM_Hb, popt_mean_intermediate[8])
m_oiii_1 = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_1, popt_mean_intermediate[9])
m_oiii_2 = gauss_lsf_flux(wave_opt_intermediate, LAM_OIII_2, popt_mean_intermediate[10])

m_ha = gauss_lsf_flux(wave_opt_intermediate, LAM_Ha, popt_mean_intermediate[11])

ax.plot(wave_opt_intermediate, m_nev, ls="--", lw=0.9, label=f"[NeV], $\sigma$ = {popt_mean_intermediate[0]/perr_mean_intermediate[0]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_oii, ls="--", lw=0.9, label=f"[OII], $\sigma$ = {popt_mean_intermediate[1]/perr_mean_intermediate[1]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_neiii, ls="--", lw=0.9, label=f"[NeIII], $\sigma$ = {popt_mean_intermediate[2]/perr_mean_intermediate[2]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_hei, ls="--", lw=0.9, label=f"He I, $\sigma$ = {popt_mean_intermediate[3]/perr_mean_intermediate[3]:.2f}", color = "#F5C242")

ax.plot(wave_opt_intermediate, m_heps, ls="--", lw=0.9, label=f"H$\epsilon$, $\sigma$ = {popt_mean_intermediate[4]/perr_mean_intermediate[4]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_hd, ls="--", lw=0.9, label=f"H$\delta$, $\sigma$ = {popt_mean_intermediate[5]/perr_mean_intermediate[5]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_hg, ls="--", lw=0.9, label=f"H$\gamma$, $\sigma$ = {popt_mean_intermediate[6]/perr_mean_intermediate[6]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_oiii_aur, ls="--", lw=0.9, label=f"[OIII]4363, $\sigma$ = {popt_mean_intermediate[7]/perr_mean_intermediate[7]:.2f}", color = "#F5C242")

ax.plot(wave_opt_intermediate, m_hb, ls="--", lw=0.9, label=f"H$\\beta$, $\sigma$ = {popt_mean_intermediate[8]/perr_mean_intermediate[8]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_oiii_1, ls="--", lw=0.9, label=f"[OIII]4959, $\sigma$ = {popt_mean_intermediate[9]/perr_mean_intermediate[9]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_oiii_2, ls="--", lw=0.9, label=f"[OIII]5007, $\sigma$ = {popt_mean_intermediate[10]/perr_mean_intermediate[10]:.2f}", color = "#F5C242")
ax.plot(wave_opt_intermediate, m_ha, ls="--", lw=0.9, label=f"H$\\alpha$, $\sigma$ = {popt_mean_intermediate[11]/perr_mean_intermediate[11]:.2f}", color = "#F5C242")

ax.axhline(0.0, color="gray", lw=0.8)
ax.set_ylabel("Normalized Flux (arbitrary units)")
lines1, labels1 = ax.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon = False, fontsize =10, ncols =2)
ax.legend(lines1, labels1, loc="upper right", frameon = False, fontsize =8, ncols =3)
#
# ax.grid()
# ax.set_title("opt continuum-subtracted mean stacks", pad=30)
ax.set_title(f"Optical Compact/Extended/Intermediate Sources Mean stacks {z_bins[0]}$<z<${z_bins[-1]}", pad=30)
# ax.set_ylim(-0.5,2)
# ax2.set_ylim(-5, 35)


# ---- bottom panel ----
resid = mean_stacked_flux_opt_compact - model_opt_mean_compact
axr.step(wave_opt_compact, resid / mean_stacked_err_opt_compact, where="mid", lw=0.9, color="#31ACC4")

resid = mean_stacked_flux_opt_extended - model_opt_mean_extended
axr.step(wave_opt_extended, resid / mean_stacked_err_opt_extended, where="mid", lw=0.9, color="firebrick")

resid = mean_stacked_flux_opt_intermediate - model_opt_mean_intermediate
axr.step(wave_opt_intermediate, resid / mean_stacked_err_opt_intermediate, where="mid", lw=0.9, color="#F5C242")

axr.axhline(0.0, color="k", lw=0.8, alpha=0.6)
axr.axhline(1.0, color="gray", ls="--", lw=0.6)
axr.axhline(-1.0, color="gray", ls="--", lw=0.6)
axr.grid(alpha= 0.3)
# axr.set_ylim(-8, 8)

plt.xlim(0.3,1)
    
axr.set_ylabel(r"$f/\sigma$")
axr.set_xlabel(r"Rest wavelength ($\mu$m)")

# ---- difference -----:
# axd.step(wave_opt_compact, mean_stacked_flux_opt_compact , where = "mid", lw = 1, color="gray",label="mean Stack - difference") 
# axd.set_ylabel(r"Flux")
# axd.set_xlabel(r"Rest wavelength ($\mu$m)")
# axd.axhline(0.0, color="gray", lw=0.8)
# axd.legend()
# axd.grid()

plt.tight_layout()
plt.show()